# 1 Packages

In [1]:
import pandas as pd ; pd.set_option('display.max_columns', None) ; pd.set_option('use_inf_as_na', True)
import numpy as np
import time
from datetime import date, timedelta
from google.colab import files
from google.colab import drive ; drive.mount('/content/drive/', force_remount=False)

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


# 2 Raw data

## 2.1 Leitura dos dados

In [2]:
# Entrada dos dados
chunks = []
for chunk in pd.read_csv('/content/drive/My Drive/Universidade Estadual do Oeste do Paraná/Dados/srag_22_08_2022.csv',
                         sep=',', encoding='latin-1', low_memory=False, skip_blank_lines=True, error_bad_lines=False, chunksize=1000000,
                         #usecols=
                         ):
    chunks.append(chunk)
df = pd.concat(chunks)

<ipython-input-2-777ff3b821c7>:3: FutureWarning: The error_bad_lines argument has been deprecated and will be removed in a future version. Use on_bad_lines in the future.


  for chunk in pd.read_csv('/content/drive/My Drive/Universidade Estadual do Oeste do Paraná/Dados/srag_22_08_2022.csv',


## 2.2 Otimização

In [3]:
# Converter colunas com ínico "DT_" para "datetime"
lista_colunas_data = [s for s in df.columns if "DT_" in s]
for colunas_data in lista_colunas_data:
    df[colunas_data] = pd.to_datetime(df[colunas_data], dayfirst=True, format='%d/%m/%Y', errors='coerce')

# Converter colunas "object" para "category"
lista_colunas_objeto = df.select_dtypes('object').columns
df[lista_colunas_objeto] = df[lista_colunas_objeto].astype('category')

# Remover variáveis não utilizadas
df.drop(['Unnamed: 0','OUTRO_SIN','OUTRO_DES','FATOR_RISC','OUT_MORBI','MORB_DESC','CLASSI_OUT','RAIOX_OUT','DT_OBITO'], axis=1, inplace=True)

## 2.3 Filtro

In [4]:
# Filtrar notificações realizadas entre jan/2009 e dez/2021
df = df[(df['DT_NOTIFIC'] >= '2009-01-01') & (df['DT_NOTIFIC'] <= '2021-12-31')].copy()

# 3 Clean data

## 3.1 Padronização dos nomes das variáveis

In [5]:
df.rename({'ID_MN_RESI':'CO_MUN_RES_1',
           'CO_MUN_RES':'CO_MUN_RES_2',
           'ID_MUNICIP':'CO_MUN_NOT',
           'CO_MU_INTE':'CO_MUN_INT'}, axis=1, inplace=True)

## 3.2 Ordenar e selecionar as variáveis

In [6]:
########### ORDENAR ###########
# Variáveis gerais
# Obrigatórias ou automáticas
variaveis_gerais = ['DT_NOTIFIC','CO_MUN_NOT','SG_UF_NOT','DT_DIGITA']

# Variáveis Identificação
# As variáveis sobre o município de residência ('CO_MUN_RES_1' e 'CO_MUN_RES_2') serão substituidas pela 'CO_MUN_RES' posteriormente.
variaveis_identificacao = ['DT_NASC','CS_SEXO','CS_GESTANT','CS_RACA','CS_ESCOL_N','CS_ZONA','VACINA','HISTO_VGM','CO_MUN_RES_1','CO_MUN_RES_2']

# Variáveis Sintomas
variaveis_sintomas = ['DT_SIN_PRI','FEBRE','TOSSE','CALAFRIO','DISPNEIA','GARGANTA','ARTRALGIA','MIALGIA','CONJUNTIV','CORIZA','DIARREIA','DOR_ABD','FADIGA','PERD_OLFT','PERD_PALA','VOMITO','DESC_RESP','SATURACAO']

# Variáveis Fatores de risco
variaveis_risco = ['CARDIOPATI','PNEUMOPATI','RENAL','HEMOGLOBI','IMUNODEPRE','TABAGISMO','METABOLICA','HEPATICA','NEUROLOGIC','OBESIDADE','PUERPERA','SIND_DOWN','HEMATOLOGI','ASMA','DIABETES']

# Variáveis Hospitalares e Laboratoriais
# As variáveis sobre a realização do PCR ('PCR_RES', 'PCR' e 'PCR_RESUL') serão substituidas pela 'REALIZADO_PCR' posteriormente.
variaveis_hosp_lab = ['HOSPITAL','DT_INTERNA','CO_MUN_INT','DT_RAIOX','RAIOX_RES','ANTIVIRAL','DT_ANTIVIR','TP_ANTIVIR','UTI','DT_ENTUTI','DT_SAIDUTI','SUPORT_VEN','PCR_RES','PCR','PCR_RESUL','DT_COLETA','DT_PCR']

# Variáveis Investigação epidemiológica
variaveis_invest_epi = ['CLASSI_FIN','CRITERIO','EVOLUCAO','DT_EVOLUCA','DT_ENCERRA']

# Variáveis Adicionais
# Utilizadas nos testes estatísticos, serão criadas posteriormente.
variaveis_adicionais = ['PANDEMIA','REGIAO_NOT','PORTE']


########### SELECIONAR ###########
df = df[variaveis_gerais +
        variaveis_identificacao +
        variaveis_sintomas +
        variaveis_risco +
        variaveis_hosp_lab +
        variaveis_invest_epi].copy()

## Variáveis: Gerais

In [7]:
########### NU_ANO ###########
# Criar variável ano de notificação
df['NU_ANO'] = df['DT_NOTIFIC'].dt.year


########### NU_MES ###########
# Criar variável mês de notificação
df['NU_MES'] = df['DT_NOTIFIC'].dt.month


########### SG_UF_NOT ###########
# Decodificar valores e padronizar da Unidade Federativa (UF) onde foi realizada a notificação de código do IGBE para sigla da UF.
df['SG_UF_NOT'] = df['SG_UF_NOT'].replace(['12'],'AC')
df['SG_UF_NOT'] = df['SG_UF_NOT'].replace(['27'],'AL')
df['SG_UF_NOT'] = df['SG_UF_NOT'].replace(['13'],'AM')
df['SG_UF_NOT'] = df['SG_UF_NOT'].replace(['16'],'AP')
df['SG_UF_NOT'] = df['SG_UF_NOT'].replace(['29'],'BA')
df['SG_UF_NOT'] = df['SG_UF_NOT'].replace(['23'],'CE')
df['SG_UF_NOT'] = df['SG_UF_NOT'].replace(['53'],'DF')
df['SG_UF_NOT'] = df['SG_UF_NOT'].replace(['32'],'ES')
df['SG_UF_NOT'] = df['SG_UF_NOT'].replace(['52'],'GO')
df['SG_UF_NOT'] = df['SG_UF_NOT'].replace(['21'],'MA')
df['SG_UF_NOT'] = df['SG_UF_NOT'].replace(['31'],'MG')
df['SG_UF_NOT'] = df['SG_UF_NOT'].replace(['50'],'MS')
df['SG_UF_NOT'] = df['SG_UF_NOT'].replace(['51'],'MT')
df['SG_UF_NOT'] = df['SG_UF_NOT'].replace(['15'],'PA')
df['SG_UF_NOT'] = df['SG_UF_NOT'].replace(['25'],'PB')
df['SG_UF_NOT'] = df['SG_UF_NOT'].replace(['26'],'PE')
df['SG_UF_NOT'] = df['SG_UF_NOT'].replace(['22'],'PI')
df['SG_UF_NOT'] = df['SG_UF_NOT'].replace(['41'],'PR')
df['SG_UF_NOT'] = df['SG_UF_NOT'].replace(['33'],'RJ')
df['SG_UF_NOT'] = df['SG_UF_NOT'].replace(['24'],'RN')
df['SG_UF_NOT'] = df['SG_UF_NOT'].replace(['11'],'RO')
df['SG_UF_NOT'] = df['SG_UF_NOT'].replace(['14'],'RR')
df['SG_UF_NOT'] = df['SG_UF_NOT'].replace(['43'],'RS')
df['SG_UF_NOT'] = df['SG_UF_NOT'].replace(['42'],'SC')
df['SG_UF_NOT'] = df['SG_UF_NOT'].replace(['28'],'SE')
df['SG_UF_NOT'] = df['SG_UF_NOT'].replace(['35'],'SP')
df['SG_UF_NOT'] = df['SG_UF_NOT'].replace(['17'],'TO')
df['SG_UF_NOT'] = df['SG_UF_NOT'].astype('category')

########### CO_MUN_NOT ###########
# A variável sobre o município onde foi realizada a notificação, denominada 'CO_MUN_NOT', apresentou dois formatos: 1. Código do município & 2. Nome do município
# Optou-se pela padronização pelo código, contudo no período do estudo (2009-2021) houve mudança no nome dos municípios, portanto foi realizado o tratamento dos nomes
# para a vinculação de dados adicionais realizada posteriormente.
df['CO_MUN_NOT'] = df['CO_MUN_NOT'].str.replace("'",'').astype('category') # Remover aspas
df['CO_MUN_NOT'] = df['CO_MUN_NOT'].str.replace("MOJI MIRIM","MOJI-MIRIM") # Adaptação do nome devido incompatibilidade com lista de municípios do IBGE
df['CO_MUN_NOT'] = df['CO_MUN_NOT'].str.replace("EMBU DAS ARTES","EMBU") # Adaptação do nome devido incompatibilidade com lista de municípios do IBGE
df['CO_MUN_NOT'] = df['CO_MUN_NOT'].str.replace("SANTA IZABEL DO PARA","SANTA ISABEL DO PARA") # Adaptação do nome devido incompatibilidade com lista de municípios do IBGE
df['CO_MUN_NOT'] = df['CO_MUN_NOT'].str.replace("BALNEARIO PICARRAS","PICARRAS") # Adaptação do nome devido incompatibilidade com lista de municípios do IBGE
df.loc[(df['CO_MUN_NOT'] == "SAO DOMINGOS") & (df['SG_UF_NOT'] == "PB"),'CO_MUN_NOT'] = 'SAO DOMINGOS DE POMBAL' # Adaptação do nome devido incompatibilidade com lista de municípios do IBGE
df.loc[(df['CO_MUN_NOT'] == "272940") & (df['SG_UF_NOT'] == "AL"),'CO_MUN_NOT'] = '292940' # Código do IBGE 272940 não existe, como municípios de residência eram BA alterou-se para 292940
df.loc[(df['CO_MUN_NOT'] == "292940") & (df['SG_UF_NOT'] == "AL"),'SG_UF_NOT'] = 'BA' # Código do IBGE 272940 não existe, como municípios de residência eram BA alterou-se para 292940

## Variáveis: Identificação

In [8]:
########### CS_SEXO ###########
# Decodificar valores
df['CS_SEXO'] = df['CS_SEXO'].astype(str)
df['CS_SEXO'].mask(df['CS_SEXO'] == "F", 'Feminino', inplace=True)
df['CS_SEXO'].mask(df['CS_SEXO'] == "M", 'Masculino', inplace=True)
df['CS_SEXO'].mask(df['CS_SEXO'] == "I", 'Ignorado', inplace=True)
df['CS_SEXO'] = df['CS_SEXO'].astype('category')


########### CS_SEXO ###########
# Decodificar valores
df['CS_GESTANT'].mask(df['CS_GESTANT'] == 1, '1º Trimestre', inplace=True)
df['CS_GESTANT'].mask(df['CS_GESTANT'] == 2, '2º Trimestre', inplace=True)
df['CS_GESTANT'].mask(df['CS_GESTANT'] == 3, '3º Trimestre', inplace=True)
df['CS_GESTANT'].mask(df['CS_GESTANT'] == 4, 'Idade Gestacional Ignorada', inplace=True)
df['CS_GESTANT'].mask(df['CS_GESTANT'] == 5, 'Não', inplace=True)
df['CS_GESTANT'].mask(df['CS_GESTANT'] == 6, 'Não se aplica', inplace=True)
df['CS_GESTANT'].mask(df['CS_GESTANT'] == 9, 'Ignorado', inplace=True)
df['CS_GESTANT'].mask(df['CS_GESTANT'] == 0, 'Ignorado', inplace=True)
df['CS_GESTANT'].mask(df['CS_GESTANT'].isnull(), 'Em branco', inplace=True)
df['CS_GESTANT'] = df['CS_GESTANT'].astype('category')


########### CS_RACA ###########
# Decodificar valores
df['CS_RACA'].mask(df['CS_RACA'] == 1, 'Branca', inplace=True)
df['CS_RACA'].mask(df['CS_RACA'] == 2, 'Preta', inplace=True)
df['CS_RACA'].mask(df['CS_RACA'] == 3, 'Amarela', inplace=True)
df['CS_RACA'].mask(df['CS_RACA'] == 4, 'Parda', inplace=True)
df['CS_RACA'].mask(df['CS_RACA'] == 5, 'Indígena', inplace=True)
df['CS_RACA'].mask(df['CS_RACA'] == 9, 'Ignorado', inplace=True)
df['CS_RACA'].mask(df['CS_RACA'].isnull(), 'Em branco', inplace=True)
df['CS_RACA'] = df['CS_RACA'].astype('category')


########### CS_ESCOL_N ###########
# Como houve mudança nas opções de preenchimento na variável sobre a escolaridade do paciente aplicou-se o versionamento da notificação segundo ano de notificação
# Decodificar valores conforme versionamento
# 2009 - 2012
df.loc[(df['NU_ANO'].isin([2009,2010,2011,2012])) & (df['CS_ESCOL_N'] == 0),'CS_ESCOL_N'] = 'Sem escolaridade'
df.loc[(df['NU_ANO'].isin([2009,2010,2011,2012])) & (df['CS_ESCOL_N'] == 1),'CS_ESCOL_N'] = 'Fundamental (1-9 anos)'
df.loc[(df['NU_ANO'].isin([2009,2010,2011,2012])) & (df['CS_ESCOL_N'] == 2),'CS_ESCOL_N'] = 'Fundamental (1-9 anos)'
df.loc[(df['NU_ANO'].isin([2009,2010,2011,2012])) & (df['CS_ESCOL_N'] == 3),'CS_ESCOL_N'] = 'Fundamental (1-9 anos)'
df.loc[(df['NU_ANO'].isin([2009,2010,2011,2012])) & (df['CS_ESCOL_N'] == 4),'CS_ESCOL_N'] = 'Fundamental (1-9 anos)'
df.loc[(df['NU_ANO'].isin([2009,2010,2011,2012])) & (df['CS_ESCOL_N'] == 5),'CS_ESCOL_N'] = 'Médio (1-3 anos)'
df.loc[(df['NU_ANO'].isin([2009,2010,2011,2012])) & (df['CS_ESCOL_N'] == 6),'CS_ESCOL_N'] = 'Médio (1-3 anos)'
df.loc[(df['NU_ANO'].isin([2009,2010,2011,2012])) & (df['CS_ESCOL_N'] == 7),'CS_ESCOL_N'] = 'Superior'
df.loc[(df['NU_ANO'].isin([2009,2010,2011,2012])) & (df['CS_ESCOL_N'] == 8),'CS_ESCOL_N'] = 'Superior'
df.loc[(df['NU_ANO'].isin([2009,2010,2011,2012])) & (df['CS_ESCOL_N'] == 9),'CS_ESCOL_N'] = 'Ignorado'
df.loc[(df['NU_ANO'].isin([2009,2010,2011,2012])) & (df['CS_ESCOL_N'] == 10),'CS_ESCOL_N'] = 'Não se aplica'

# 2013 - 2018
df.loc[(df['NU_ANO'].isin([2013,2014,2015,2016,2017,2018])) & (df['CS_ESCOL_N'] == 0),'CS_ESCOL_N'] = 'Sem escolaridade'
df.loc[(df['NU_ANO'].isin([2013,2014,2015,2016,2017,2018])) & (df['CS_ESCOL_N'] == 1),'CS_ESCOL_N'] = 'Fundamental (1-9 anos)'
df.loc[(df['NU_ANO'].isin([2013,2014,2015,2016,2017,2018])) & (df['CS_ESCOL_N'] == 2),'CS_ESCOL_N'] = 'Médio (1-3 anos)'
df.loc[(df['NU_ANO'].isin([2013,2014,2015,2016,2017,2018])) & (df['CS_ESCOL_N'] == 3),'CS_ESCOL_N'] = 'Superior'
df.loc[(df['NU_ANO'].isin([2013,2014,2015,2016,2017,2018])) & (df['CS_ESCOL_N'] == 9),'CS_ESCOL_N'] = 'Ignorado'
df.loc[(df['NU_ANO'].isin([2013,2014,2015,2016,2017,2018])) & (df['CS_ESCOL_N'] == 10),'CS_ESCOL_N'] = 'Não se aplica'
df.loc[(df['NU_ANO'].isin([2013,2014,2015,2016,2017,2018])) & (df['CS_ESCOL_N'] == 5),'CS_ESCOL_N'] = 'Ignorado' # Não descrito no dicionário de dados

# 2019 - 2021
df.loc[(df['NU_ANO'].isin([2019,2020,2021,2022])) & (df['CS_ESCOL_N'] == 0),'CS_ESCOL_N'] = 'Sem escolaridade'
df.loc[(df['NU_ANO'].isin([2019,2020,2021,2022])) & (df['CS_ESCOL_N'] == 1),'CS_ESCOL_N'] = 'Fundamental (1-9 anos)'
df.loc[(df['NU_ANO'].isin([2019,2020,2021,2022])) & (df['CS_ESCOL_N'] == 2),'CS_ESCOL_N'] = 'Fundamental (1-9 anos)'
df.loc[(df['NU_ANO'].isin([2019,2020,2021,2022])) & (df['CS_ESCOL_N'] == 3),'CS_ESCOL_N'] = 'Médio (1-3 anos)'
df.loc[(df['NU_ANO'].isin([2019,2020,2021,2022])) & (df['CS_ESCOL_N'] == 4),'CS_ESCOL_N'] = 'Superior'
df.loc[(df['NU_ANO'].isin([2019,2020,2021,2022])) & (df['CS_ESCOL_N'] == 5),'CS_ESCOL_N'] = 'Não se aplica'
df.loc[(df['NU_ANO'].isin([2019,2020,2021,2022])) & (df['CS_ESCOL_N'] == 9),'CS_ESCOL_N'] = 'Ignorado'
df.loc[(df['NU_ANO'].isin([2019,2020,2021,2022])) & (df['CS_ESCOL_N'] == 10),'CS_ESCOL_N'] = 'Ignorado' # Não descrito no dicionário de dados

df['CS_ESCOL_N'].mask(df['CS_ESCOL_N'].isnull(),'Em branco', inplace=True)
df['CS_ESCOL_N'] = df['CS_ESCOL_N'].astype('category')


########### CS_ZONA ###########
# Campo inserido em 2019, em uso de 2019 a 2021
# Decodificar valores
df['CS_ZONA'].mask(df['CS_ZONA'] == 1, 'Urbana', inplace=True)
df['CS_ZONA'].mask(df['CS_ZONA'] == 2, 'Rural', inplace=True)
df['CS_ZONA'].mask(df['CS_ZONA'] == 3, 'Periurbana', inplace=True)
df['CS_ZONA'].mask(df['CS_ZONA'] == 9, 'Ignorado', inplace=True)
df['CS_ZONA'].mask(df['CS_ZONA'].isnull(),'Em branco', inplace=True)
df['CS_ZONA'] = df['CS_ZONA'].astype('category')


########### VACINA ###########
# Decodificar valores
df['VACINA'].mask(df['VACINA'] == 1, 'Sim', inplace=True)
df['VACINA'].mask(df['VACINA'] == 2, 'Não', inplace=True)
df['VACINA'].mask(df['VACINA'] == 9, 'Ignorado', inplace=True)
df['VACINA'] = df['VACINA'].astype('category')


########### HISTO_VGM ###########
# Histórico de viagem - Paciente tem histórico de viagem internacional até 14 dias antes do início dos sintomas?
# Campo inserido em 2019, em uso de 2019 a 2021
# Decodificar valores
df['HISTO_VGM'].mask(df['HISTO_VGM'] == 1, 'Sim', inplace=True)
df['HISTO_VGM'].mask(df['HISTO_VGM'] == 2, 'Não', inplace=True)
df['HISTO_VGM'].mask(df['HISTO_VGM'] == 9, 'Ignorado', inplace=True)
df['HISTO_VGM'].mask(df['HISTO_VGM'] == 0, 'Em branco', inplace=True)
df['HISTO_VGM'].mask(df['HISTO_VGM'].isnull(),'Em branco', inplace=True)
df['HISTO_VGM'] = df['HISTO_VGM'].astype('category')


########### CO_MUN_RES ###########
# As informações sobre o código do município onde foi realizada a notificação está dividida em duas variáveis: 'CO_MUN_RES_2' e 'CO_MUN_RES_1'
# Caso a variável 'CO_MUN_RES_2' for nula será preenchida com a variável 'CO_MUN_RES_1'
df['CO_MUN_RES'] = np.where(df['CO_MUN_RES_2'].isnull(), df['CO_MUN_RES_1'], df['CO_MUN_RES_2']).astype(float)

# Substituir valores nulos para '000000'
df['CO_MUN_RES'] = df['CO_MUN_RES'].fillna(000000)

# Mudar formato para string e substituir valores '000000' para 'Em branco'
df['CO_MUN_RES'] = df['CO_MUN_RES'].astype(int).astype(str).str.replace('000000',"Em branco")

# Remover variáveis não utilizadas
df.drop(['CO_MUN_RES_1','CO_MUN_RES_2'], axis=1, inplace=True)

## Variáveis: Sintomas

In [9]:
########### FEBRE ###########
# Decodificar valores
df['FEBRE'].mask(df['FEBRE'] == 1, 'Sim', inplace=True)
df['FEBRE'].mask(df['FEBRE'] == 2, 'Não', inplace=True)
df['FEBRE'].mask(df['FEBRE'] == 9, 'Ignorado', inplace=True)
df['FEBRE'].mask(df['FEBRE'].isnull(),'Em branco', inplace=True)
df['FEBRE'] = df['FEBRE'].astype('category')


########### TOSSE ###########
# Decodificar valores
df['TOSSE'].mask(df['TOSSE'] == 1, 'Sim', inplace=True)
df['TOSSE'].mask(df['TOSSE'] == 2, 'Não', inplace=True)
df['TOSSE'].mask(df['TOSSE'] == 9, 'Ignorado', inplace=True)
df['TOSSE'].mask(df['TOSSE'].isnull(),'Em branco', inplace=True)
df['TOSSE'] = df['TOSSE'].astype('category')


########### DISPNEIA ###########
# Decodificar valores
df['DISPNEIA'].mask(df['DISPNEIA'] == 1, 'Sim', inplace=True)
df['DISPNEIA'].mask(df['DISPNEIA'] == 2, 'Não', inplace=True)
df['DISPNEIA'].mask(df['DISPNEIA'] == 9, 'Ignorado', inplace=True)
df['DISPNEIA'].mask(df['DISPNEIA'].isnull(),'Em branco', inplace=True)
df['DISPNEIA'] = df['DISPNEIA'].astype('category')


########### GARGANTA ###########
# Decodificar valores
df['GARGANTA'].mask(df['GARGANTA'] == 1, 'Sim', inplace=True)
df['GARGANTA'].mask(df['GARGANTA'] == 2, 'Não', inplace=True)
df['GARGANTA'].mask(df['GARGANTA'] == 9, 'Ignorado', inplace=True)
df['GARGANTA'].mask(df['GARGANTA'].isnull(),'Em branco', inplace=True)
df['GARGANTA'] = df['GARGANTA'].astype('category')


########### MIALGIA ###########
# Campo removido em 2018, em uso de 2009 a 2018
# Decodificar valores
df['MIALGIA'].mask(df['MIALGIA'] == 1, 'Sim', inplace=True)
df['MIALGIA'].mask(df['MIALGIA'] == 2, 'Não', inplace=True)
df['MIALGIA'].mask(df['MIALGIA'] == 9, 'Ignorado', inplace=True)
df['MIALGIA'].mask(df['MIALGIA'].isnull(),'Em branco', inplace=True)
df['MIALGIA'] = df['MIALGIA'].astype('category')


########### CALAFRIO ###########
# Campo removido em 2012, em uso de 2009 a 2012
# Decodificar valores
df['CALAFRIO'].mask(df['CALAFRIO'] == 1, 'Sim', inplace=True)
df['CALAFRIO'].mask(df['CALAFRIO'] == 2, 'Não', inplace=True)
df['CALAFRIO'].mask(df['CALAFRIO'] == 9, 'Ignorado', inplace=True)
df['CALAFRIO'].mask(df['CALAFRIO'].isnull(),'Em branco', inplace=True)
df['CALAFRIO'] = df['CALAFRIO'].astype('category')


########### ARTRALGIA ###########
# Campo removido em 2012, em uso de 2009 a 2012
# Decodificar valores
df['ARTRALGIA'].mask(df['ARTRALGIA'] == 1, 'Sim', inplace=True)
df['ARTRALGIA'].mask(df['ARTRALGIA'] == 2, 'Não', inplace=True)
df['ARTRALGIA'].mask(df['ARTRALGIA'] == 9, 'Ignorado', inplace=True)
df['ARTRALGIA'].mask(df['ARTRALGIA'].isnull(),'Em branco', inplace=True)
df['ARTRALGIA'] = df['ARTRALGIA'].astype('category')


########### CONJUNTIV ###########
# Campo removido em 2012, em uso de 2009 a 2012
# Decodificar valores
df['CONJUNTIV'].mask(df['CONJUNTIV'] == 1, 'Sim', inplace=True)
df['CONJUNTIV'].mask(df['CONJUNTIV'] == 2, 'Não', inplace=True)
df['CONJUNTIV'].mask(df['CONJUNTIV'] == 9, 'Ignorado', inplace=True)
df['CONJUNTIV'].mask(df['CONJUNTIV'].isnull(),'Em branco', inplace=True)
df['CONJUNTIV'] = df['CONJUNTIV'].astype('category')


########### CORIZA ###########
# Campo removido em 2012, em uso de 2009 a 2012
# Decodificar valores
df['CORIZA'].mask(df['CORIZA'] == 1, 'Sim', inplace=True)
df['CORIZA'].mask(df['CORIZA'] == 2, 'Não', inplace=True)
df['CORIZA'].mask(df['CORIZA'] == 9, 'Ignorado', inplace=True)
df['CORIZA'].mask(df['CORIZA'].isnull(),'Em branco', inplace=True)
df['CORIZA'] = df['CORIZA'].astype('category')


########### DIARREIA ###########
# Campo removido em 2012 e inserido novamente em 2019, em uso entre 2009 a 2012 e também em 2019 e 2021
# Decodificar valores
df['DIARREIA'].mask(df['DIARREIA'] == 1, 'Sim', inplace=True)
df['DIARREIA'].mask(df['DIARREIA'] == 2, 'Não', inplace=True)
df['DIARREIA'].mask(df['DIARREIA'] == 9, 'Ignorado', inplace=True)
df['DIARREIA'].mask(df['DIARREIA'].isnull(),'Em branco', inplace=True)
df['DIARREIA'] = df['DIARREIA'].astype('category')


########### DESC_RESP ###########
# Campo inserido em 2013, em uso de 2013 a 2021
# Decodificar valores
df['DESC_RESP'].mask(df['DESC_RESP'] == 1, 'Sim', inplace=True)
df['DESC_RESP'].mask(df['DESC_RESP'] == 2, 'Não', inplace=True)
df['DESC_RESP'].mask(df['DESC_RESP'] == 9, 'Ignorado', inplace=True)
df['DESC_RESP'].mask(df['DESC_RESP'].isnull(),'Em branco', inplace=True)
df['DESC_RESP'] = df['DESC_RESP'].astype('category')


########### SATURACAO ###########
# Campo inserido em 2013, em uso de 2013 a 2021
# Decodificar valores
df['SATURACAO'].mask(df['SATURACAO'] == 1, 'Sim', inplace=True)
df['SATURACAO'].mask(df['SATURACAO'] == 2, 'Não', inplace=True)
df['SATURACAO'].mask(df['SATURACAO'] == 9, 'Ignorado', inplace=True)
df['SATURACAO'].mask(df['SATURACAO'].isnull(),'Em branco', inplace=True)
df['SATURACAO'] = df['SATURACAO'].astype('category')


########### VOMITO ###########
# Campo inserido em 2019, em uso de 2019 a 2021
# Decodificar valores
df['VOMITO'].mask(df['VOMITO'] == 1, 'Sim', inplace=True)
df['VOMITO'].mask(df['VOMITO'] == 2, 'Não', inplace=True)
df['VOMITO'].mask(df['VOMITO'] == 9, 'Ignorado', inplace=True)
df['VOMITO'].mask(df['VOMITO'].isnull(),'Em branco', inplace=True)
df['VOMITO'] = df['VOMITO'].astype('category')


########### DOR_ABD ###########
# Campo inserido em 2020, em uso de 2020 a 2021
# Decodificar valores
df['DOR_ABD'].mask(df['DOR_ABD'] == 1, 'Sim', inplace=True)
df['DOR_ABD'].mask(df['DOR_ABD'] == 2, 'Não', inplace=True)
df['DOR_ABD'].mask(df['DOR_ABD'] == 9, 'Ignorado', inplace=True)
df['DOR_ABD'].mask(df['DOR_ABD'].isnull(),'Em branco', inplace=True)
df['DOR_ABD'] = df['DOR_ABD'].astype('category')


########### FADIGA ###########
# Campo inserido em 2020, em uso de 2020 a 2021
# Decodificar valores
df['FADIGA'].mask(df['FADIGA'] == 1, 'Sim', inplace=True)
df['FADIGA'].mask(df['FADIGA'] == 2, 'Não', inplace=True)
df['FADIGA'].mask(df['FADIGA'] == 9, 'Ignorado', inplace=True)
df['FADIGA'].mask(df['FADIGA'].isnull(),'Em branco', inplace=True)
df['FADIGA'] = df['FADIGA'].astype('category')


########### PERD_OLFT ###########
# Campo inserido em 2020, em uso de 2020 a 2021
# Decodificar valores
df['PERD_OLFT'].mask(df['PERD_OLFT'] == 1, 'Sim', inplace=True)
df['PERD_OLFT'].mask(df['PERD_OLFT'] == 2, 'Não', inplace=True)
df['PERD_OLFT'].mask(df['PERD_OLFT'] == 9, 'Ignorado', inplace=True)
df['PERD_OLFT'].mask(df['PERD_OLFT'].isnull(),'Em branco', inplace=True)
df['PERD_OLFT'] = df['PERD_OLFT'].astype('category')


########### PERD_PALA ###########
# Campo inserido em 2020, em uso de 2020 a 2021
# Decodificar valores
df['PERD_PALA'].mask(df['PERD_PALA'] == 1, 'Sim', inplace=True)
df['PERD_PALA'].mask(df['PERD_PALA'] == 2, 'Não', inplace=True)
df['PERD_PALA'].mask(df['PERD_PALA'] == 9, 'Ignorado', inplace=True)
df['PERD_PALA'].mask(df['PERD_PALA'].isnull(),'Em branco', inplace=True)
df['PERD_PALA'] = df['PERD_PALA'].astype('category')

## Variáveis: Comorbidades

In [10]:
########### CARDIOPATI ###########
# Decodificar valores
df['CARDIOPATI'].mask(df['CARDIOPATI'] == 1, 'Sim', inplace=True)
df['CARDIOPATI'].mask(df['CARDIOPATI'] == 2, 'Não', inplace=True)
df['CARDIOPATI'].mask(df['CARDIOPATI'] == 9, 'Ignorado', inplace=True)
df['CARDIOPATI'].mask(df['CARDIOPATI'].isnull(),'Em branco', inplace=True)
df['CARDIOPATI'] = df['CARDIOPATI'].astype('category')


########### PNEUMOPATI ###########
# Decodificar valores
df['PNEUMOPATI'].mask(df['PNEUMOPATI'] == 1, 'Sim', inplace=True)
df['PNEUMOPATI'].mask(df['PNEUMOPATI'] == 2, 'Não', inplace=True)
df['PNEUMOPATI'].mask(df['PNEUMOPATI'] == 9, 'Ignorado', inplace=True)
df['PNEUMOPATI'].mask(df['PNEUMOPATI'].isnull(),'Em branco', inplace=True)
df['PNEUMOPATI'] = df['PNEUMOPATI'].astype('category')


########### RENAL ###########
# Decodificar valores
df['RENAL'].mask(df['RENAL'] == 1, 'Sim', inplace=True)
df['RENAL'].mask(df['RENAL'] == 2, 'Não', inplace=True)
df['RENAL'].mask(df['RENAL'] == 9, 'Ignorado', inplace=True)
df['RENAL'].mask(df['RENAL'].isnull(),'Em branco', inplace=True)
df['RENAL'] = df['RENAL'].astype('category')


########### IMUNODEPRE ###########
# Decodificar valores
df['IMUNODEPRE'].mask(df['IMUNODEPRE'] == 1, 'Sim', inplace=True)
df['IMUNODEPRE'].mask(df['IMUNODEPRE'] == 2, 'Não', inplace=True)
df['IMUNODEPRE'].mask(df['IMUNODEPRE'] == 9, 'Ignorado', inplace=True)
df['IMUNODEPRE'].mask(df['IMUNODEPRE'].isnull(),'Em branco', inplace=True)
df['IMUNODEPRE'] = df['IMUNODEPRE'].astype('category')



########### HEMOGLOBI ###########
# Campo inserido em 2009, em uso de 2009 a 2012
# Decodificar valores
df['HEMOGLOBI'].mask(df['HEMOGLOBI'] == 1, 'Sim', inplace=True)
df['HEMOGLOBI'].mask(df['HEMOGLOBI'] == 2, 'Não', inplace=True)
df['HEMOGLOBI'].mask(df['HEMOGLOBI'] == 9, 'Ignorado', inplace=True)
df['HEMOGLOBI'].mask(df['HEMOGLOBI'].isnull(),'Em branco', inplace=True)
df['HEMOGLOBI'] = df['HEMOGLOBI'].astype('category')


########### TABAGISMO ###########
# Campo inserido em 2009, em uso de 2009 a 2012
# Decodificar valores
df['TABAGISMO'].mask(df['TABAGISMO'] == 1, 'Sim', inplace=True)
df['TABAGISMO'].mask(df['TABAGISMO'] == 2, 'Não', inplace=True)
df['TABAGISMO'].mask(df['TABAGISMO'] == 9, 'Ignorado', inplace=True)
df['TABAGISMO'].mask(df['TABAGISMO'].isnull(),'Em branco', inplace=True)
df['TABAGISMO'] = df['TABAGISMO'].astype('category')


########### METABOLICA ###########
# Campo inserido em 2009, em uso de 2009 a 2012
# Decodificar valores
df['METABOLICA'].mask(df['METABOLICA'] == 1, 'Sim', inplace=True)
df['METABOLICA'].mask(df['METABOLICA'] == 2, 'Não', inplace=True)
df['METABOLICA'].mask(df['METABOLICA'] == 9, 'Ignorado', inplace=True)
df['METABOLICA'].mask(df['METABOLICA'].isnull(),'Em branco', inplace=True)
df['METABOLICA'] = df['METABOLICA'].astype('category')


########### HEPATICA ###########
# Campo inserido em 2013, em uso de 2013 a 2021
# Decodificar valores
df['HEPATICA'].mask(df['HEPATICA'] == 1, 'Sim', inplace=True)
df['HEPATICA'].mask(df['HEPATICA'] == 2, 'Não', inplace=True)
df['HEPATICA'].mask(df['HEPATICA'] == 9, 'Ignorado', inplace=True)
df['HEPATICA'].mask(df['HEPATICA'].isnull(),'Em branco', inplace=True)
df['HEPATICA'] = df['HEPATICA'].astype('category')


########### NEUROLOGIC ###########
# Campo inserido em 2013, em uso de 2013 a 2021
# Decodificar valores
df['NEUROLOGIC'].mask(df['NEUROLOGIC'] == 1, 'Sim', inplace=True)
df['NEUROLOGIC'].mask(df['NEUROLOGIC'] == 2, 'Não', inplace=True)
df['NEUROLOGIC'].mask(df['NEUROLOGIC'] == 9, 'Ignorado', inplace=True)
df['NEUROLOGIC'].mask(df['NEUROLOGIC'].isnull(),'Em branco', inplace=True)
df['NEUROLOGIC'] = df['NEUROLOGIC'].astype('category')


########### SIND_DOWN ###########
# Campo inserido em 2013, em uso de 2013 a 2021
# Decodificar valores
df['SIND_DOWN'].mask(df['SIND_DOWN'] == 1, 'Sim', inplace=True)
df['SIND_DOWN'].mask(df['SIND_DOWN'] == 2, 'Não', inplace=True)
df['SIND_DOWN'].mask(df['SIND_DOWN'] == 9, 'Ignorado', inplace=True)
df['SIND_DOWN'].mask(df['SIND_DOWN'].isnull(),'Em branco', inplace=True)
df['SIND_DOWN'] = df['SIND_DOWN'].astype('category')


########### PUERPERA ###########
# Campo inserido em 2013, em uso de 2013 a 2021
# Decodificar valores
df['PUERPERA'].mask(df['PUERPERA'] == 1, 'Sim', inplace=True)
df['PUERPERA'].mask(df['PUERPERA'] == 2, 'Não', inplace=True)
df['PUERPERA'].mask(df['PUERPERA'] == 9, 'Ignorado', inplace=True)
df['PUERPERA'].mask(df['PUERPERA'].isnull(),'Em branco', inplace=True)
df['PUERPERA'] = df['PUERPERA'].astype('category')


########### OBESIDADE ###########
# Campo inserido em 2013, em uso de 2013 a 2021
# Decodificar valores
df['OBESIDADE'].mask(df['OBESIDADE'] == 1, 'Sim', inplace=True)
df['OBESIDADE'].mask(df['OBESIDADE'] == 2, 'Não', inplace=True)
df['OBESIDADE'].mask(df['OBESIDADE'] == 9, 'Ignorado', inplace=True)
df['OBESIDADE'].mask(df['OBESIDADE'].isnull(),'Em branco', inplace=True)
df['OBESIDADE'] = df['OBESIDADE'].astype('category')


########### HEMATOLOGI ###########
# Campo inserido em 2019, em uso de 2019 a 2021
# Decodificar valores
df['HEMATOLOGI'].mask(df['HEMATOLOGI'] == 1, 'Sim', inplace=True)
df['HEMATOLOGI'].mask(df['HEMATOLOGI'] == 2, 'Não', inplace=True)
df['HEMATOLOGI'].mask(df['HEMATOLOGI'] == 9, 'Ignorado', inplace=True)
df['HEMATOLOGI'].mask(df['HEMATOLOGI'].isnull(),'Em branco', inplace=True)
df['HEMATOLOGI'] = df['HEMATOLOGI'].astype('category')


########### ASMA ###########
# Campo inserido em 2019, em uso de 2019 a 2021
# Decodificar valores
df['ASMA'].mask(df['ASMA'] == 1, 'Sim', inplace=True)
df['ASMA'].mask(df['ASMA'] == 2, 'Não', inplace=True)
df['ASMA'].mask(df['ASMA'] == 9, 'Ignorado', inplace=True)
df['ASMA'].mask(df['ASMA'].isnull(),'Em branco', inplace=True)
df['ASMA'] = df['ASMA'].astype('category')


########### DIABETES ###########
# Campo inserido em 2019, em uso de 2019 a 2021
# Decodificar valores
df['DIABETES'].mask(df['DIABETES'] == 1, 'Sim', inplace=True)
df['DIABETES'].mask(df['DIABETES'] == 2, 'Não', inplace=True)
df['DIABETES'].mask(df['DIABETES'] == 9, 'Ignorado', inplace=True)
df['DIABETES'].mask(df['DIABETES'].isnull(),'Em branco', inplace=True)
df['DIABETES'] = df['DIABETES'].astype('category')

## Variáveis: Hospitalar e Laboratorial

In [11]:
########### HOSPITAL ###########
# Todo período
# Decodificar valores
df['HOSPITAL'].mask(df['HOSPITAL'] == 1, 'Sim', inplace=True)
df['HOSPITAL'].mask(df['HOSPITAL'] == 2, 'Não', inplace=True)
df['HOSPITAL'].mask(df['HOSPITAL'] == 9, 'Ignorado', inplace=True)
df['HOSPITAL'].mask(df['HOSPITAL'].isnull(),'Em branco', inplace=True)
df['HOSPITAL'] = df['HOSPITAL'].astype('category')


########### DT_INTERNA ###########
# Todo período


########### CO_MUN_INT ###########
# Todo período
# Tratar valores
df['CO_MUN_INT'] = df['CO_MUN_INT'].fillna(999999).astype(int).astype(str)
df['CO_MUN_INT'].mask(df['CO_MUN_INT'] == '999999', 'Em branco', inplace=True)


########### RAIOX_RES ###########
# Todo período
# Decodificar valores
df['RAIOX_RES'].mask(df['RAIOX_RES'] == 1, 'Normal', inplace=True)
df['RAIOX_RES'].mask(df['RAIOX_RES'] == 2, 'Infiltrado intersticial', inplace=True)
df['RAIOX_RES'].mask(df['RAIOX_RES'] == 3, 'Consolidação', inplace=True)
df['RAIOX_RES'].mask(df['RAIOX_RES'] == 4, 'Misto', inplace=True)
df['RAIOX_RES'].mask(df['RAIOX_RES'] == 5, 'Outro', inplace=True)
df['RAIOX_RES'].mask(df['RAIOX_RES'] == 6, 'Não realizado', inplace=True)
df['RAIOX_RES'].mask(df['RAIOX_RES'] == 9, 'Ignorado', inplace=True)
df['RAIOX_RES'].mask(df['RAIOX_RES'].isnull(),'Em branco', inplace=True)
df['RAIOX_RES'] = df['RAIOX_RES'].astype('category')


########### DT_RAIOX ###########
# Todo período


########### ANTIVIRAL ###########
# Campo inserido em 2013, em uso de 2013 a 2021
# Decodificar valores conforme versionamento
# 2009 a 2012
df.loc[(df['NU_ANO'].isin([2009,2010,2011,2012])) & (df['ANTIVIRAL'] == 1),'ANTIVIRAL'] = 'Não' # No dicionário de dados = "Não"
df.loc[(df['NU_ANO'].isin([2009,2010,2011,2012])) & (df['ANTIVIRAL'] == 2),'ANTIVIRAL'] = 'Sim' # No dicionário de dados = "Oseltamivir"
df.loc[(df['NU_ANO'].isin([2009,2010,2011,2012])) & (df['ANTIVIRAL'] == 3),'ANTIVIRAL'] = 'Sim' # No dicionário de dados = "Zanamivir"
df.loc[(df['NU_ANO'].isin([2009,2010,2011,2012])) & (df['ANTIVIRAL'] == 4),'ANTIVIRAL'] = 'Sim' # No dicionário de dados = "Outro"
df.loc[(df['NU_ANO'].isin([2009,2010,2011,2012])) & (df['ANTIVIRAL'] == 9),'ANTIVIRAL'] = 'Ignorado' # No dicionário de dados = "Ignorado"

# 2013 a 2018
df.loc[(df['NU_ANO'].isin([2013,2014,2015,2016,2017,2018])) & (df['ANTIVIRAL'] == 1),'ANTIVIRAL'] = 'Não' # No dicionário de dados = "Não"
df.loc[(df['NU_ANO'].isin([2013,2014,2015,2016,2017,2018])) & (df['ANTIVIRAL'] == 2),'ANTIVIRAL'] = 'Sim' # No dicionário de dados = "Oseltamivir"
df.loc[(df['NU_ANO'].isin([2013,2014,2015,2016,2017,2018])) & (df['ANTIVIRAL'] == 3),'ANTIVIRAL'] = 'Sim' # No dicionário de dados = "Zanamivir"
df.loc[(df['NU_ANO'].isin([2013,2014,2015,2016,2017,2018])) & (df['ANTIVIRAL'] == 4),'ANTIVIRAL'] = 'Sim' # No dicionário de dados = "Outro"
df.loc[(df['NU_ANO'].isin([2013,2014,2015,2016,2017,2018])) & (df['ANTIVIRAL'] == 9),'ANTIVIRAL'] = 'Ignorado' # No dicionário de dados = "Ignorado"

# 2019 a 2021
df.loc[(df['NU_ANO'].isin([2019,2020,2021])) & (df['ANTIVIRAL'] == 1),'ANTIVIRAL'] = 'Sim'
df.loc[(df['NU_ANO'].isin([2019,2020,2021])) & (df['ANTIVIRAL'] == 2),'ANTIVIRAL'] = 'Não'
df.loc[(df['NU_ANO'].isin([2019,2020,2021])) & (df['ANTIVIRAL'] == 3),'ANTIVIRAL'] = 'Sim' # Baseado nos dicionários de dados anteriores
df.loc[(df['NU_ANO'].isin([2019,2020,2021])) & (df['ANTIVIRAL'] == 4),'ANTIVIRAL'] = 'Sim' # Baseado nos dicionários de dados anteriores
df.loc[(df['NU_ANO'].isin([2019,2020,2021])) & (df['ANTIVIRAL'] == 9),'ANTIVIRAL'] = 'Ignorado'

df['ANTIVIRAL'].mask(df['ANTIVIRAL'].isnull(),'Em branco', inplace=True)
df['ANTIVIRAL'] = df['ANTIVIRAL'].astype('category')


########### TP_ANTIVIR ###########
# Campo inserido em 2019, em uso de 2019 a 2021
# Decodificar valores
df.loc[(df['NU_ANO'].isin([2018,2019,2020,2021])) & (df['TP_ANTIVIR'] == 1),'TP_ANTIVIR'] = 'Oseltamivir'
df.loc[(df['NU_ANO'].isin([2018,2019,2020,2021])) & (df['TP_ANTIVIR'] == 2),'TP_ANTIVIR'] = 'Zanamivir'
df.loc[(df['NU_ANO'].isin([2018,2019,2020,2021])) & (df['TP_ANTIVIR'] == 3),'TP_ANTIVIR'] = 'Outro'
df['TP_ANTIVIR'].mask(df['TP_ANTIVIR'].isnull(),'Em branco', inplace=True)
df['TP_ANTIVIR'] = df['TP_ANTIVIR'].astype('category')


########### SUPORT_VEN ###########
# Campo inserido em 2013, em uso de 2013 a 2021
# Decodificar valores
df['SUPORT_VEN'].mask(df['SUPORT_VEN'] == 1, 'Sim, invasivo', inplace=True)
df['SUPORT_VEN'].mask(df['SUPORT_VEN'] == 2, 'Sim, não invasivo', inplace=True)
df['SUPORT_VEN'].mask(df['SUPORT_VEN'] == 3, 'Não', inplace=True)
df['SUPORT_VEN'].mask(df['SUPORT_VEN'] == 9, 'Ignorado', inplace=True)
df['SUPORT_VEN'].mask(df['SUPORT_VEN'].isnull(),'Em branco', inplace=True)
df['SUPORT_VEN'] = df['SUPORT_VEN'].astype('category')


########### UTI ###########
# Campo inserido em 2013, em uso de 2013 a 2021
# Decodificar valores
df['UTI'].mask(df['UTI'] == 1, 'Sim', inplace=True)
df['UTI'].mask(df['UTI'] == 2, 'Não', inplace=True)
df['UTI'].mask(df['UTI'] == 9, 'Ignorado', inplace=True)
df['UTI'].mask(df['UTI'].isnull(),'Em branco', inplace=True)
df['UTI'] = df['UTI'].astype('category')


########### DT_ENTUTI ###########
# Campo inserido em 2013, em uso de 2013 a 2021


########### DT_SAIDUTI ###########
# Campo inserido em 2013, em uso de 2013 a 2021


########### PCR_REALIZADO ###########
# Os dados sobre a realização do exame laboratorial por método PCR estão distribuidos em 3 variáveis ('PCR_RES', 'PCR' e 'PCR_RESULT').
# Para realizar a união de tais dados numa única variável foi realizada a codificação conforme versão da notificação.
# Criar coluna para verificar se exame laboratorial por método PCR foi realizado
# 2009 - 2012
df.loc[(df['NU_ANO'].isin([2009,2010,2011,2012])) & (df['PCR_RES'] == 1),'PCR_REALIZADO'] = 'Sim'  # No dicionário de dados = "Positivo"
df.loc[(df['NU_ANO'].isin([2009,2010,2011,2012])) & (df['PCR_RES'] == 2),'PCR_REALIZADO'] = 'Sim' # No dicionário de dados = "Negativo"
df.loc[(df['NU_ANO'].isin([2009,2010,2011,2012])) & (df['PCR_RES'] == 3),'PCR_REALIZADO'] = 'Sim' # No dicionário de dados = "Inconclusivo"
df.loc[(df['NU_ANO'].isin([2009,2010,2011,2012])) & (df['PCR_RES'] == 4),'PCR_REALIZADO'] = 'Não' # No dicionário de dados = "Não realizado"
df.loc[(df['NU_ANO'].isin([2009,2010,2011,2012])) & (df['PCR_RES'].isnull()),'PCR_REALIZADO'] = 'Em branco'

# 2013 - 2018
df.loc[(df['NU_ANO'].isin([2013,2014,2015,2016,2017,2018,2019])) & (df['PCR'] == 1),'PCR_REALIZADO'] = 'Sim' # No dicionário de dados = "Sim"
df.loc[(df['NU_ANO'].isin([2013,2014,2015,2016,2017,2018,2019])) & (df['PCR'] == 2),'PCR_REALIZADO'] = 'Não' # No dicionário de dados = "Não"
df.loc[(df['NU_ANO'].isin([2013,2014,2015,2016,2017,2018,2019])) & (df['PCR'] == 9),'PCR_REALIZADO'] = 'Ignorado' # No dicionário de dados = "Ignorado"
df.loc[(df['NU_ANO'].isin([2013,2014,2015,2016,2017,2018,2019])) & (df['PCR'].isnull()),'PCR_REALIZADO'] = 'Em branco'

# 2019 - 2021
df.loc[(df['NU_ANO'].isin([2018,2019,2020,2021])) & (df['PCR_RESUL'] == 1),'PCR_REALIZADO'] = 'Sim' # No dicionário de dados = "Detectável"
df.loc[(df['NU_ANO'].isin([2018,2019,2020,2021])) & (df['PCR_RESUL'] == 2),'PCR_REALIZADO'] = 'Sim' # No dicionário de dados = "Não Detectável"
df.loc[(df['NU_ANO'].isin([2018,2019,2020,2021])) & (df['PCR_RESUL'] == 3),'PCR_REALIZADO'] = 'Sim' # No dicionário de dados = "Inconclusivo"
df.loc[(df['NU_ANO'].isin([2018,2019,2020,2021])) & (df['PCR_RESUL'] == 4),'PCR_REALIZADO'] = 'Não' # No dicionário de dados = "Não Realizado"
df.loc[(df['NU_ANO'].isin([2018,2019,2020,2021])) & (df['PCR_RESUL'] == 5),'PCR_REALIZADO'] = 'Sim' # No dicionário de dados = "Aguardando Resultado"
df.loc[(df['NU_ANO'].isin([2018,2019,2020,2021])) & (df['PCR_RESUL'] == 9),'PCR_REALIZADO'] = 'Ignorado' # No dicionário de dados = "Ignorado"
df.loc[(df['NU_ANO'].isin([2018,2019,2020,2021])) & (df['PCR_RESUL'].isnull()),'PCR_REALIZADO'] = 'Em branco'

df['PCR_REALIZADO'] = df['PCR_REALIZADO'].astype('category')
df.drop(['PCR_RES','PCR','PCR_RESUL'], axis=1, inplace=True) # Remover variáveis não utilizadas


########### DT_COLETA ###########
# Campo inserido em 2013, em uso de 2013 a 2021


########### DT_PCR ###########
# Campo inserido em 2013, em uso de 2013 a 2021

## Variáveis: Investigação epidemiológica

In [12]:
########### CLASSI_FIN ###########
# Decodificar valores
# 2009 - 2012
df.loc[(df['NU_ANO'].isin([2009,2010,2011,2012])) & (df['CLASSI_FIN'] == 1),'CLASSI_FIN'] = 'Influenza' # No dicionário de dados = "Influenza por novo subtipo viral"
df.loc[(df['NU_ANO'].isin([2009,2010,2011,2012])) & (df['CLASSI_FIN'] == 2),'CLASSI_FIN'] = 'Outro agente etiológico' # No dicionário de dados = "Outro agente infeccioso"
df.loc[(df['NU_ANO'].isin([2009,2010,2011,2012])) & (df['CLASSI_FIN'] == 3),'CLASSI_FIN'] = 'Não especificado' # No dicionário de dados = "Descartado"

# 2013 - 2018
df.loc[(df['NU_ANO'].isin([2013,2014,2015,2016,2017,2018])) & (df['CLASSI_FIN'] == 1),'CLASSI_FIN'] = 'Influenza' # No dicionário de dados = "SRAG por Influenza"
df.loc[(df['NU_ANO'].isin([2013,2014,2015,2016,2017,2018])) & (df['CLASSI_FIN'] == 2),'CLASSI_FIN'] = 'Outro agente etiológico' # No dicionário de dados = "SRAG por outros vírus respiratórios"
df.loc[(df['NU_ANO'].isin([2013,2014,2015,2016,2017,2018])) & (df['CLASSI_FIN'] == 3),'CLASSI_FIN'] = 'Outro agente etiológico' # No dicionário de dados = "SRAG por outros agentes etiológicos"

# 2019 - 2021
df.loc[(df['NU_ANO'].isin([2019,2020,2021,2022])) & (df['CLASSI_FIN'] == 1),'CLASSI_FIN'] = 'Influenza' # No dicionário de dados = "SRAG por Influenza"
df.loc[(df['NU_ANO'].isin([2019,2020,2021,2022])) & (df['CLASSI_FIN'] == 2),'CLASSI_FIN'] = 'Outro agente etiológico' # No dicionário de dados = "SRAG por outros vírus respiratórios"
df.loc[(df['NU_ANO'].isin([2019,2020,2021,2022])) & (df['CLASSI_FIN'] == 3),'CLASSI_FIN'] = 'Outro agente etiológico' # No dicionário de dados = "SRAG por outros agentes etiológicos"
df.loc[(df['NU_ANO'].isin([2019,2020,2021,2022])) & (df['CLASSI_FIN'] == 5),'CLASSI_FIN'] = 'COVID-19' # No dicionário de dados = "SRAG por COVID-19"

df['CLASSI_FIN'].mask(df['CLASSI_FIN'] == 4,'Não especificado', inplace=True)
df['CLASSI_FIN'].mask(df['CLASSI_FIN'] == 9,'Ignorado', inplace=True)
df['CLASSI_FIN'].mask(df['CLASSI_FIN'].isnull(),'Em branco', inplace=True)
df['CLASSI_FIN'] = df['CLASSI_FIN'].astype('category')


########### CRITERIO ###########
# Decodificar valores
# 2009 - 2018
df.loc[(df['NU_ANO'].isin([2009,2010,2011,2012,2013,2014,2015,2016,2017,2018])) & (df['CRITERIO'] == 1),'CRITERIO'] = 'Laboratorial'
df.loc[(df['NU_ANO'].isin([2009,2010,2011,2012,2013,2014,2015,2016,2017,2018])) & (df['CRITERIO'] == 2),'CRITERIO'] = 'Clínico-Epidemiológico'
df.loc[(df['NU_ANO'].isin([2009,2010,2011,2012,2013,2014,2015,2016,2017,2018])) & (df['CRITERIO'] == 3),'CRITERIO'] = 'Clínico'

# 2019 - 2021
df.loc[(df['NU_ANO'].isin([2019,2020,2021,2022])) & (df['CRITERIO'] == 1),'CRITERIO'] = 'Laboratorial'
df.loc[(df['NU_ANO'].isin([2019,2020,2021,2022])) & (df['CRITERIO'] == 2),'CRITERIO'] = 'Clínico-Epidemiológico'
df.loc[(df['NU_ANO'].isin([2019,2020,2021,2022])) & (df['CRITERIO'] == 3),'CRITERIO'] = 'Clínico'
df.loc[(df['NU_ANO'].isin([2019,2020,2021,2022])) & (df['CRITERIO'] == 4),'CRITERIO'] = 'Clínico-Imagem'

df['CRITERIO'].mask(df['CRITERIO'].isnull(),'Em branco', inplace=True)
df['CRITERIO'] = df['CRITERIO'].astype('category')


########### EVOLUCAO ###########
# Decodificar valores
# 2009 - 2012
df.loc[(df['NU_ANO'].isin([2009,2010,2011,2012])) & (df['EVOLUCAO'] == 1),'EVOLUCAO'] = 'Cura'
df.loc[(df['NU_ANO'].isin([2009,2010,2011,2012])) & (df['EVOLUCAO'] == 2),'EVOLUCAO'] = 'Óbito'
df.loc[(df['NU_ANO'].isin([2009,2010,2011,2012])) & (df['EVOLUCAO'] == 3),'EVOLUCAO'] = 'Óbito por outras causas'
df.loc[(df['NU_ANO'].isin([2009,2010,2011,2012])) & (df['EVOLUCAO'] == 4),'EVOLUCAO'] = 'Óbito em investigação'
df.loc[(df['NU_ANO'].isin([2009,2010,2011,2012])) & (df['EVOLUCAO'] == 9),'EVOLUCAO'] = 'Ignorado'

# 2013 - 2018
df.loc[(df['NU_ANO'].isin([2013,2014,2015,2016,2017,2018])) & (df['EVOLUCAO'] == 1),'EVOLUCAO'] = 'Cura'
df.loc[(df['NU_ANO'].isin([2013,2014,2015,2016,2017,2018])) & (df['EVOLUCAO'] == 2),'EVOLUCAO'] = 'Óbito'

# 2019 - 2021
df.loc[(df['NU_ANO'].isin([2019,2020,2021,2022])) & (df['EVOLUCAO'] == 1),'EVOLUCAO'] = 'Cura'
df.loc[(df['NU_ANO'].isin([2019,2020,2021,2022])) & (df['EVOLUCAO'] == 2),'EVOLUCAO'] = 'Óbito'
df.loc[(df['NU_ANO'].isin([2019,2020,2021,2022])) & (df['EVOLUCAO'] == 3),'EVOLUCAO'] = 'Óbito por outras causas'

df['EVOLUCAO'].mask(df['EVOLUCAO'] == 9,'Ignorado', inplace=True)
df['EVOLUCAO'].mask(df['EVOLUCAO'].isnull(),'Em branco', inplace=True)
df['EVOLUCAO'] = df['EVOLUCAO'].astype('category')


########### DT_EVOLUCA ###########
# Campo inserido em 2019, em uso de 2019 a 2021


########### DT_ENCERRA ###########
# Todo período

## Variáveis: Adicionais

In [13]:
########### NU_IDADE_N ###########
# Criar variável idade
df['NU_IDADE_N'] = (df['DT_NOTIFIC'] - df['DT_NASC']) // timedelta(days=365.2425)


########### GRUPO_IDADE ###########
# Criar variável grupo de idade
df.loc[df['NU_IDADE_N'] >= 60, 'GRUPO_IDADE'] = '60 anos ou mais'
df.loc[df['NU_IDADE_N'] <60, 'GRUPO_IDADE'] = '30 a 59 anos'
df.loc[df['NU_IDADE_N'] <30, 'GRUPO_IDADE'] = '0 a 29 anos'
df['GRUPO_IDADE'] = df['GRUPO_IDADE'].astype('category')


########### PANDEMIA ###########
df['PANDEMIA'] = np.where(df['DT_NOTIFIC'] >= "2020-02-03","Durante","Anterior")
df['PANDEMIA'] = df['PANDEMIA'].astype('category')


########### REGIAO_NOT ###########
# Criar a variável sobre a região do Brasil onde foi realizada a notificação
df.loc[df['SG_UF_NOT'] == 'AC',		'REGIAO_NOT'] = 'Norte'
df.loc[df['SG_UF_NOT'] == 'AL',		'REGIAO_NOT'] = 'Nordeste'
df.loc[df['SG_UF_NOT'] == 'AM',		'REGIAO_NOT'] = 'Norte'
df.loc[df['SG_UF_NOT'] == 'AP',		'REGIAO_NOT'] = 'Norte'
df.loc[df['SG_UF_NOT'] == 'BA',		'REGIAO_NOT'] = 'Nordeste'
df.loc[df['SG_UF_NOT'] == 'CE',		'REGIAO_NOT'] = 'Nordeste'
df.loc[df['SG_UF_NOT'] == 'DF',	  'REGIAO_NOT'] = 'Centro-Oeste'
df.loc[df['SG_UF_NOT'] == 'ES',	  'REGIAO_NOT'] = 'Sudeste'
df.loc[df['SG_UF_NOT'] == 'GO',		'REGIAO_NOT'] = 'Centro-Oeste'
df.loc[df['SG_UF_NOT'] == 'MA',		'REGIAO_NOT'] = 'Nordeste'
df.loc[df['SG_UF_NOT'] == 'MG',	  'REGIAO_NOT'] = 'Sudeste'
df.loc[df['SG_UF_NOT'] == 'MS',	  'REGIAO_NOT'] = 'Centro-Oeste'
df.loc[df['SG_UF_NOT'] == 'MT',	  'REGIAO_NOT'] = 'Centro-Oeste'
df.loc[df['SG_UF_NOT'] == 'PA',		'REGIAO_NOT'] = 'Norte'
df.loc[df['SG_UF_NOT'] == 'PB',		'REGIAO_NOT'] = 'Nordeste'
df.loc[df['SG_UF_NOT'] == 'PE',		'REGIAO_NOT'] = 'Nordeste'
df.loc[df['SG_UF_NOT'] == 'PI',		'REGIAO_NOT'] = 'Nordeste'
df.loc[df['SG_UF_NOT'] == 'PR',		'REGIAO_NOT'] = 'Sul'
df.loc[df['SG_UF_NOT'] == 'RJ',	  'REGIAO_NOT'] = 'Sudeste'
df.loc[df['SG_UF_NOT'] == 'RN',   'REGIAO_NOT'] = 'Nordeste'
df.loc[df['SG_UF_NOT'] == 'RO',		'REGIAO_NOT'] = 'Norte'
df.loc[df['SG_UF_NOT'] == 'RR',		'REGIAO_NOT'] = 'Norte'
df.loc[df['SG_UF_NOT'] == 'RS',	  'REGIAO_NOT'] = 'Sul'
df.loc[df['SG_UF_NOT'] == 'SC',	  'REGIAO_NOT'] = 'Sul'
df.loc[df['SG_UF_NOT'] == 'SE',		'REGIAO_NOT'] = 'Nordeste'
df.loc[df['SG_UF_NOT'] == 'SP',		'REGIAO_NOT'] = 'Sudeste'
df.loc[df['SG_UF_NOT'] == 'TO',		'REGIAO_NOT'] = 'Norte'
df['REGIAO_NOT'] = df['REGIAO_NOT'].astype('category')


########### VARIÁVEIS MUNICIPAIS: FRONTEIRA E PORTE ###########
# Criar a chave composta (Município de notificação + UF de notificação) para união dos dados
# Utilizada a chave composta para evitar união de informações de municipíos com nomes homônimos
df['KEY_MUN_UF_NOT'] = df['CO_MUN_NOT'].astype(str) + " (" + df['SG_UF_NOT'].astype(str) + ")"
df['KEY_MUN_UF_NOT'] = df['KEY_MUN_UF_NOT'].astype('category')

# Entrada das inforamções municipais
municipios = pd.read_csv('https://docs.google.com/spreadsheets/d/e/2PACX-1vTWHR7fPRibraW2O7bUyR25CoWEw6oSg7Y1mN3BcJBHj0wX2lPvLTtoA1XR4BvTZULsL-pj7sfwYlLk/pub?gid=2095536398&single=true&output=csv',
                         sep=',', encoding='utf-8', low_memory=False, skip_blank_lines=True, error_bad_lines=False,
                         usecols=['ID_MUNICIP_UF','IBGE','FAIXA_FRONTEIRA','FRONT_AEREA','FRONTEIRA_MARITIMA','NENHUMA_FRONTEIRA','PORTE']).astype('category').copy()

# Unir informações municipais aos registros individuais
df = df.set_index('KEY_MUN_UF_NOT').join(municipios.set_index('ID_MUNICIP_UF'))

# Substituir nomes por códigos do IBGE
df['CO_MUN_NOT'] = df['IBGE']
df['CO_MUN_NOT'] = df['CO_MUN_NOT'].astype('category')

# Refazer index
df = df.reset_index(drop=True)


########### FRONTEIRA ###########
df.loc[(df['FRONTEIRA_MARITIMA']==0) & (df['FRONT_AEREA']==0) & (df['FAIXA_FRONTEIRA']==0),'FRONTEIRA'] = 'Não'
df.loc[(df['FRONTEIRA_MARITIMA']==1) & (df['FRONT_AEREA']==0) & (df['FAIXA_FRONTEIRA']==0),'FRONTEIRA'] = 'Sim'
df.loc[(df['FRONTEIRA_MARITIMA']==0) & (df['FRONT_AEREA']==1) & (df['FAIXA_FRONTEIRA']==0),'FRONTEIRA'] = 'Sim'
df.loc[(df['FRONTEIRA_MARITIMA']==0) & (df['FRONT_AEREA']==0) & (df['FAIXA_FRONTEIRA']==1),'FRONTEIRA'] = 'Sim'
df.loc[(df['FRONTEIRA_MARITIMA']==1) & (df['FRONT_AEREA']==1) & (df['FAIXA_FRONTEIRA']==0),'FRONTEIRA'] = 'Sim'
df.loc[(df['FRONTEIRA_MARITIMA']==1) & (df['FRONT_AEREA']==0) & (df['FAIXA_FRONTEIRA']==1),'FRONTEIRA'] = 'Sim'
df.loc[(df['FRONTEIRA_MARITIMA']==0) & (df['FRONT_AEREA']==1) & (df['FAIXA_FRONTEIRA']==1),'FRONTEIRA'] = 'Sim'
df.loc[(df['FRONTEIRA_MARITIMA']==1) & (df['FRONT_AEREA']==1) & (df['FAIXA_FRONTEIRA']==1),'FRONTEIRA'] = 'Sim'
df['FRONTEIRA'] = df['FRONTEIRA'].astype('category')


# Remover variáveis não utilizadas
df.drop(['IBGE','FRONTEIRA_MARITIMA','FRONT_AEREA','FAIXA_FRONTEIRA','NENHUMA_FRONTEIRA'], axis=1, inplace=True)

<ipython-input-13-3d20e4f53295>:58: FutureWarning: The error_bad_lines argument has been deprecated and will be removed in a future version. Use on_bad_lines in the future.


  municipios = pd.read_csv('https://docs.google.com/spreadsheets/d/e/2PACX-1vTWHR7fPRibraW2O7bUyR25CoWEw6oSg7Y1mN3BcJBHj0wX2lPvLTtoA1XR4BvTZULsL-pj7sfwYlLk/pub?gid=2095536398&single=true&output=csv',


## Ordenar variáveis

In [14]:
df = df[['DT_NOTIFIC','NU_ANO','NU_MES','DT_DIGITA','PANDEMIA','CO_MUN_NOT','SG_UF_NOT','REGIAO_NOT','PORTE','FRONTEIRA',
    'CO_MUN_RES','DT_NASC','NU_IDADE_N','GRUPO_IDADE','CS_SEXO','CS_GESTANT','CS_RACA','CS_ESCOL_N','CS_ZONA','VACINA','HISTO_VGM',
    'DT_SIN_PRI','FEBRE','TOSSE','CALAFRIO','DISPNEIA','GARGANTA','ARTRALGIA','MIALGIA','CONJUNTIV','CORIZA','DIARREIA','DOR_ABD','FADIGA','PERD_OLFT','PERD_PALA','VOMITO','DESC_RESP','SATURACAO',
    'CARDIOPATI','PNEUMOPATI','RENAL','HEMOGLOBI','IMUNODEPRE','TABAGISMO','METABOLICA','HEPATICA','NEUROLOGIC','OBESIDADE','PUERPERA','SIND_DOWN','HEMATOLOGI','ASMA','DIABETES',
    'HOSPITAL','DT_INTERNA','CO_MUN_INT','RAIOX_RES','DT_RAIOX','ANTIVIRAL','DT_ANTIVIR','TP_ANTIVIR','UTI','DT_ENTUTI','DT_SAIDUTI','SUPORT_VEN','PCR_REALIZADO','DT_COLETA','DT_PCR',
    'CLASSI_FIN','CRITERIO','EVOLUCAO','DT_EVOLUCA','DT_ENCERRA']].sort_values(by=['DT_NOTIFIC'], ascending=True)

# 4 Evaluation (Completeness and Timeliness)

## 4.1 Completitude

In [15]:
VALOR_CORTE = 70

In [16]:
########### COMPL_IDENT ###########
########### COMPL_IDENT ###########
########### COMPL_IDENT ###########
# Avaliar completitude das variáveis sobre identificação, se percentual de completitude >70% (Boa) se <70% (Ruim)
# Versão 1 (7 variáveis sobre identificação)
#completitude_identificacao_versao_1 = [2009,2010,2011,2012,2013,2014,2015,2016,2017,2018]
df.loc[(df['DT_NOTIFIC'] >= '2009-01-01') & (df['DT_NOTIFIC'] <= '2018-12-31'),'COMPL_IDENT'] = (round(100-((df[["DT_NASC","CS_SEXO","CS_GESTANT","CS_RACA","CS_ESCOL_N","VACINA","CO_MUN_RES"]].isna().sum(axis=1) + # Testar se variáveis sobre datas são nulas
                                                                                                             df[["DT_NASC","CS_SEXO","CS_GESTANT","CS_RACA","CS_ESCOL_N","VACINA","CO_MUN_RES"]].eq('Ignorado').sum(axis=1) + # Testar se variáveis categóricas são "Ignorado"
                                                                                                             df[["DT_NASC","CS_SEXO","CS_GESTANT","CS_RACA","CS_ESCOL_N","VACINA","CO_MUN_RES"]].eq('Em branco').sum(axis=1)) # Testar se variáveis categóricas são "Em branco"
                                                                                                             / 7) * 100, # Calcular percentual de completitude
                                                                                                       2) > VALOR_CORTE).replace({True:1,False:0}).astype('category') # Testar se percentual de completitude é >70% ou <70%

# Versão 2 (8 variáveis sobre identificação)
#completitude_identificacao_versao_2 = [2019]
df.loc[(df['DT_NOTIFIC'] >= '2019-01-01') & (df['DT_NOTIFIC'] <= '2020-07-31'),'COMPL_IDENT'] = (round(100-((df[["DT_NASC","CS_SEXO","CS_GESTANT","CS_RACA","CS_ESCOL_N",'CS_ZONA',"VACINA","CO_MUN_RES"]].isna().sum(axis=1) + # Testar se variáveis sobre datas são nulas
                                                                                                             df[["DT_NASC","CS_SEXO","CS_GESTANT","CS_RACA","CS_ESCOL_N",'CS_ZONA',"VACINA","CO_MUN_RES"]].eq('Ignorado').sum(axis=1) + # Testar se variáveis categóricas são "Ignorado"
                                                                                                             df[["DT_NASC","CS_SEXO","CS_GESTANT","CS_RACA","CS_ESCOL_N",'CS_ZONA',"VACINA","CO_MUN_RES"]].eq('Em branco').sum(axis=1)) # Testar se variáveis categóricas são "Em branco"
                                                                                                             / 8) * 100, # Calcular percentual de completitude
                                                                                                       2) > VALOR_CORTE).replace({True:1,False:0}).astype('category') # Testar se percentual de completitude é >70% ou <70%

# Versão 3 (9 variáveis sobre identificação)
#completitude_identificacao_versao_2 = [2020,2021]
df.loc[(df['DT_NOTIFIC'] >= '2020-08-01') & (df['DT_NOTIFIC'] <= '2021-12-31'),'COMPL_IDENT'] = (round(100-((df[["DT_NASC","CS_SEXO","CS_GESTANT","CS_RACA","CS_ESCOL_N",'CS_ZONA',"VACINA","HISTO_VGM","CO_MUN_RES"]].isna().sum(axis=1) + # Testar se variáveis sobre datas são nulas
                                                                                                             df[["DT_NASC","CS_SEXO","CS_GESTANT","CS_RACA","CS_ESCOL_N",'CS_ZONA',"VACINA","HISTO_VGM","CO_MUN_RES"]].eq('Ignorado').sum(axis=1) + # Testar se variáveis categóricas são "Ignorado"
                                                                                                             df[["DT_NASC","CS_SEXO","CS_GESTANT","CS_RACA","CS_ESCOL_N",'CS_ZONA',"VACINA","HISTO_VGM","CO_MUN_RES"]].eq('Em branco').sum(axis=1)) # Testar se variáveis categóricas são "Em branco"
                                                                                                             / 9) * 100, # Calcular percentual de completitude
                                                                                                       2) > VALOR_CORTE).replace({True:1,False:0}).astype('category') # Testar se percentual de completitude é >70% ou <70%

In [17]:
########### COMPL_SINTO ###########
########### COMPL_SINTO ###########
########### COMPL_SINTO ###########
# Avaliar completitude das variáveis sobre sintomas, se percentual de completitude >70% (Boa) se <70% (Ruim)
# Versão 1 (11 variáveis sobre sintomas)
#completitude_sintomas_versao_1 = [2009,2010,2011,2012]
df.loc[(df['DT_NOTIFIC'] >= '2009-01-01') & (df['DT_NOTIFIC'] <= '2012-08-31'),'COMPL_SINTO'] = (round(100-((df[["DT_SIN_PRI","FEBRE","TOSSE","DISPNEIA","GARGANTA","MIALGIA","CALAFRIO","ARTRALGIA","CONJUNTIV","CORIZA","DIARREIA"]].isna().sum(axis=1) + # Testar se variáveis sobre datas são nulas
                                                                                                             df[["DT_SIN_PRI","FEBRE","TOSSE","DISPNEIA","GARGANTA","MIALGIA","CALAFRIO","ARTRALGIA","CONJUNTIV","CORIZA","DIARREIA"]].eq('Ignorado').sum(axis=1) + # Testar se variáveis categóricas são "Ignorado"
                                                                                                             df[["DT_SIN_PRI","FEBRE","TOSSE","DISPNEIA","GARGANTA","MIALGIA","CALAFRIO","ARTRALGIA","CONJUNTIV","CORIZA","DIARREIA"]].eq('Em branco').sum(axis=1)) # Testar se variáveis categóricas são "Em branco"
                                                                                                            / 11) * 100, # Calcular percentual de completitude
                                                                                                       2) > VALOR_CORTE).replace({True:1,False:0}).astype('category') # Testar se percentual de completitude é >70% ou <70%

# Versão 2 (8 variáveis sobre sintomas)
#completitude_sintomas_versao_2 = [2013,2014,2015,2016,2017,2018]
df.loc[(df['DT_NOTIFIC'] >= '2012-09-01') & (df['DT_NOTIFIC'] <= '2018-12-31'),'COMPL_SINTO'] = (round(100-((df[["DT_SIN_PRI","FEBRE","TOSSE","DISPNEIA","GARGANTA","MIALGIA","DESC_RESP","SATURACAO"]].isna().sum(axis=1) + # Testar se variáveis sobre datas são nulas
                                                                                                             df[["DT_SIN_PRI","FEBRE","TOSSE","DISPNEIA","GARGANTA","MIALGIA","DESC_RESP","SATURACAO"]].eq('Ignorado').sum(axis=1) + # Testar se variáveis categóricas são "Ignorado"
                                                                                                             df[["DT_SIN_PRI","FEBRE","TOSSE","DISPNEIA","GARGANTA","MIALGIA","DESC_RESP","SATURACAO"]].eq('Em branco').sum(axis=1)) # Testar se variáveis categóricas são "Em branco"
                                                                                                             / 8) * 100, # Calcular percentual de completitude
                                                                                                       2) > VALOR_CORTE).replace({True:1,False:0}).astype('category') # Testar se percentual de completitude é >70% ou <70%

# Versão 3 (9 variáveis sobre sintomas)
#completitude_sintomas_versao_3 = [2019]
df.loc[(df['DT_NOTIFIC'] >= '2019-01-01') & (df['DT_NOTIFIC'] <= '2020-07-31'),'COMPL_SINTO'] = (round(100-((df[["DT_SIN_PRI","FEBRE","TOSSE","DISPNEIA","GARGANTA","DIARREIA","DESC_RESP","SATURACAO","VOMITO"]].isna().sum(axis=1) + # Testar se variáveis sobre datas são nulas
                                                                                                             df[["DT_SIN_PRI","FEBRE","TOSSE","DISPNEIA","GARGANTA","DIARREIA","DESC_RESP","SATURACAO","VOMITO"]].eq('Ignorado').sum(axis=1) + # Testar se variáveis categóricas são "Ignorado"
                                                                                                             df[["DT_SIN_PRI","FEBRE","TOSSE","DISPNEIA","GARGANTA","DIARREIA","DESC_RESP","SATURACAO","VOMITO"]].eq('Em branco').sum(axis=1)) # Testar se variáveis categóricas são "Em branco"
                                                                                                             / 9) * 100, # Calcular percentual de completitude
                                                                                                       2) > VALOR_CORTE).replace({True:1,False:0}).astype('category') # Testar se percentual de completitude é >70% ou <70%

# Versão 4 (13 variáveis sobre sintomas)
#completitude_sintomas_versao_4 = [2020,2021]
df.loc[(df['DT_NOTIFIC'] >= '2020-08-01') & (df['DT_NOTIFIC'] <= '2021-12-31'),'COMPL_SINTO'] = (round(100-((df[["DT_SIN_PRI","FEBRE","TOSSE","DISPNEIA","GARGANTA","DIARREIA","DESC_RESP","SATURACAO","VOMITO","DOR_ABD","FADIGA","PERD_OLFT","PERD_PALA"]].isna().sum(axis=1) + # Testar se variáveis sobre datas são nulas
                                                                                                             df[["DT_SIN_PRI","FEBRE","TOSSE","DISPNEIA","GARGANTA","DIARREIA","DESC_RESP","SATURACAO","VOMITO","DOR_ABD","FADIGA","PERD_OLFT","PERD_PALA"]].eq('Ignorado').sum(axis=1) + # Testar se variáveis categóricas são "Ignorado"
                                                                                                             df[["DT_SIN_PRI","FEBRE","TOSSE","DISPNEIA","GARGANTA","DIARREIA","DESC_RESP","SATURACAO","VOMITO","DOR_ABD","FADIGA","PERD_OLFT","PERD_PALA"]].eq('Em branco').sum(axis=1)) # Testar se variáveis categóricas são "Em branco"
                                                                                                             / 13) * 100, # Calcular percentual de completitude
                                                                                                       2) > VALOR_CORTE).replace({True:1,False:0}).astype('category') # Testar se percentual de completitude é >70% ou <70%

In [18]:
########### COMPL_RISCO ###########
########### COMPL_RISCO ###########
########### COMPL_RISCO ###########
# Avaliar completitude das variáveis sobre fatores de risco, se percentual de completitude >70% (Boa) se <70% (Ruim)

# Versão 1 (7 variáveis sobre fatores de risco)
df.loc[(df['DT_NOTIFIC'] >= '2009-01-01') & (df['DT_NOTIFIC'] <= '2012-08-31'),'COMPL_RISCO'] = (round(100-((df[['CARDIOPATI','PNEUMOPATI','RENAL','IMUNODEPRE','HEMOGLOBI','TABAGISMO','METABOLICA']].isna().sum(axis=1) + # Testar se variáveis sobre datas são nulas
                                                                                                             df[['CARDIOPATI','PNEUMOPATI','RENAL','IMUNODEPRE','HEMOGLOBI','TABAGISMO','METABOLICA']].eq('Ignorado').sum(axis=1) + # Testar se variáveis categóricas são "Ignorado"
                                                                                                             df[['CARDIOPATI','PNEUMOPATI','RENAL','IMUNODEPRE','HEMOGLOBI','TABAGISMO','METABOLICA']].eq('Em branco').sum(axis=1)) # Testar se variáveis categóricas são "Em branco"
                                                                                                            / 7) * 100, # Calcular percentual de completitude
                                                                                                       2) > VALOR_CORTE).replace({True:1,False:0}).astype('category') # Testar se percentual de completitude é >70% ou <70%

# Versão 2 (9 variáveis sobre fatores de risco)
df.loc[(df['DT_NOTIFIC'] >= '2012-09-01') & (df['DT_NOTIFIC'] <= '2018-12-31'),'COMPL_RISCO'] = (round(100-((df[['CARDIOPATI','PNEUMOPATI','RENAL','IMUNODEPRE','HEPATICA','NEUROLOGIC','SIND_DOWN','PUERPERA','OBESIDADE']].isna().sum(axis=1) + # Testar se variáveis sobre datas são nulas
                                                                                                             df[['CARDIOPATI','PNEUMOPATI','RENAL','IMUNODEPRE','HEPATICA','NEUROLOGIC','SIND_DOWN','PUERPERA','OBESIDADE']].eq('Ignorado').sum(axis=1) + # Testar se variáveis categóricas são "Ignorado"
                                                                                                             df[['CARDIOPATI','PNEUMOPATI','RENAL','IMUNODEPRE','HEPATICA','NEUROLOGIC','SIND_DOWN','PUERPERA','OBESIDADE']].eq('Em branco').sum(axis=1)) # Testar se variáveis categóricas são "Em branco"
                                                                                                             / 9) * 100, # Calcular percentual de completitude
                                                                                                       2) > VALOR_CORTE).replace({True:1,False:0}).astype('category') # Testar se percentual de completitude é >70% ou <70%

# Versão 3 (12 variáveis sobre fatores de risco)
df.loc[(df['DT_NOTIFIC'] >= '2019-01-01') & (df['DT_NOTIFIC'] <= '2021-12-31'),'COMPL_RISCO'] = (round(100-((df[['CARDIOPATI','PNEUMOPATI','RENAL','IMUNODEPRE','HEPATICA','NEUROLOGIC','SIND_DOWN','PUERPERA','OBESIDADE','HEMATOLOGI','ASMA','DIABETES']].isna().sum(axis=1) + # Testar se variáveis sobre datas são nulas
                                                                                                             df[['CARDIOPATI','PNEUMOPATI','RENAL','IMUNODEPRE','HEPATICA','NEUROLOGIC','SIND_DOWN','PUERPERA','OBESIDADE','HEMATOLOGI','ASMA','DIABETES']].eq('Ignorado').sum(axis=1) + # Testar se variáveis categóricas são "Ignorado"
                                                                                                             df[['CARDIOPATI','PNEUMOPATI','RENAL','IMUNODEPRE','HEPATICA','NEUROLOGIC','SIND_DOWN','PUERPERA','OBESIDADE','HEMATOLOGI','ASMA','DIABETES']].eq('Em branco').sum(axis=1)) # Testar se variáveis categóricas são "Em branco"
                                                                                                             / 12) * 100, # Calcular percentual de completitude
                                                                                                       2) > VALOR_CORTE).replace({True:1,False:0}).astype('category') # Testar se percentual de completitude é >70% ou <70%

In [19]:
########### COMPL_HOSP_LAB ###########
########### COMPL_HOSP_LAB ###########
########### COMPL_HOSP_LAB ###########

# Completitude parciais das variáveis hospitalares e laboratoriais
# Como essas variáveis apresentam campos adicionais disponibilizados para preenchimento conforme variáveis prévias
# foi necessário aplicar regras condicionadas para avaliação da completitude segundo grupos de variáveis


# A completitude das 3 variáveis sobre a hospitalização ('HOSPITAL', 'DT_INTERNA' e 'CO_MUN_INT) será avaliada caso hospitalizado ('HOSPITAL' == 'Sim')
# caso contrário será avaliada apenas a variável 'HOSPITAL'
# NÚMERADOR (CONTAR CAMPOS NULOS/IGNORADOS/EM BRANCO)
df['COMPL_HOSP_LAB_HOSPITAL_NUM'] = np.select([(df['HOSPITAL'].isin(['Em branco','Ignorado'])), # 1 em branco/ignorado
                                               (df['HOSPITAL'] == "Não"), # 0 em branco/ignorado
                                               (df['HOSPITAL'] == "Sim") & (df['DT_INTERNA'].notna()) & (df['CO_MUN_INT'] != 'Em branco'), # 0 em branco/ignorado
                                               (df['HOSPITAL'] == "Sim") & (df['DT_INTERNA'].isna()) & (df['CO_MUN_INT'] != 'Em branco'), # 1 em branco/ignorado
                                               (df['HOSPITAL'] == "Sim") & (df['DT_INTERNA'].notna()) & (df['CO_MUN_INT'] == 'Em branco'), # 1 em branco/ignorado
                                               (df['HOSPITAL'] == "Sim") & (df['DT_INTERNA'].isna()) & (df['CO_MUN_INT'] == 'Em branco')], # 2 em branco/ignorado,
                                              [1, 0, 0, 1, 1, 2])
# DENOMINADOR
df['COMPL_HOSP_LAB_HOSPITAL_DEN'] = np.select([(df['HOSPITAL'].isin(['Em branco','Ignorado'])), # 1 campo
                                               (df['HOSPITAL'] == "Não"), # 1 campo
                                               (df['HOSPITAL'] == "Sim")], # 3 campos
                                              [1, 1, 3])


# A completitude das 2 variáveis sobre raio-X ('RAIOX_RES' e 'DT_RAIOX') será avaliada caso o exame seja realizado ('RAIOX_RES' != 'Não realizado')
# caso contrário será avaliada apenas a variável 'RAIOX_RES'
# NÚMERADOR (CONTAR CAMPOS NULOS/IGNORADOS/EM BRANCO)
df['COMPL_HOSP_LAB_RAIOX_NUM'] = np.select([(df['RAIOX_RES'].isin(['Em branco','Ignorado'])), # 1 em branco/ignorado
                                            (df['RAIOX_RES'] == "Não realizado"), # 0 em branco/ignorado
                                            (df['RAIOX_RES'].isin(['Infiltrado intersticial','Outro','Normal','Consolidação','Misto'])) & (df['DT_RAIOX'].isna()), # 1 em branco/ignorado
                                            (df['RAIOX_RES'].isin(['Infiltrado intersticial','Outro','Normal','Consolidação','Misto'])) & (df['DT_RAIOX'].notna())], # 0 em branco/ignorado
                                           [1, 0, 1, 0])
# DENOMINADOR
df['COMPL_HOSP_LAB_RAIOX_DEN'] = np.select([(df['RAIOX_RES'].isin(['Em branco','Ignorado'])), # 1 campo
                                            (df['RAIOX_RES'] == "Não realizado"), # 1 campo
                                            (df['RAIOX_RES'].isin(['Infiltrado intersticial','Outro','Normal','Consolidação','Misto']))], # 2 campos
                                           [1, 1, 2])


# A completitude das 3 variáveis sobre uso de antiviral ('ANTIVIRAL', 'DT_ANTIVIR' e 'TP_ANTIVIR) será avaliada caso apresente uso de antiviral ('ANTIVIRAL' == 'Sim')
# caso contrário será avaliada apenas a variável 'ANTIVIRAL'
# NÚMERADOR (CONTAR CAMPOS NULOS/IGNORADOS/EM BRANCO)
df['COMPL_HOSP_LAB_ANTIVIRAL_NUM'] = np.select([# 2009-2012
                                                (df['DT_NOTIFIC'] >= '2009-01-01') & (df['DT_NOTIFIC'] <= '2012-08-31'), # 0 em branco/ignorado - Campo não utilizado

                                                # 2013-2018
                                                (df['DT_NOTIFIC'] >= '2012-09-01') & (df['DT_NOTIFIC'] <= '2018-12-31') & (df['ANTIVIRAL'].isin(['Em branco','Ignorado'])), # 1 em branco/ignorado
                                                (df['DT_NOTIFIC'] >= '2012-09-01') & (df['DT_NOTIFIC'] <= '2018-12-31') & (df['ANTIVIRAL'] == "Não"), # 0 em branco/ignorado
                                                (df['DT_NOTIFIC'] >= '2012-09-01') & (df['DT_NOTIFIC'] <= '2018-12-31') & (df['ANTIVIRAL'] == "Sim") & (df['DT_ANTIVIR'].isna()), # 1 em branco/ignorado
                                                (df['DT_NOTIFIC'] >= '2012-09-01') & (df['DT_NOTIFIC'] <= '2018-12-31') & (df['ANTIVIRAL'] == "Sim") & (df['DT_ANTIVIR'].notna()), # 0 em branco/ignorado

                                                # 2019-2021
                                                (df['DT_NOTIFIC'] >= '2019-01-01') & (df['DT_NOTIFIC'] <= '2021-12-31') & (df['ANTIVIRAL'].isin(['Em branco','Ignorado'])), # 1 em branco/ignorado
                                                (df['DT_NOTIFIC'] >= '2019-01-01') & (df['DT_NOTIFIC'] <= '2021-12-31') & (df['ANTIVIRAL'] == "Não"), # 0 em branco/ignorado
                                                (df['DT_NOTIFIC'] >= '2019-01-01') & (df['DT_NOTIFIC'] <= '2021-12-31') & (df['ANTIVIRAL'] == "Sim") & (df['DT_ANTIVIR'].notna()) & (df['TP_ANTIVIR'] != 'Em branco'), # 0 em branco/ignorado
                                                (df['DT_NOTIFIC'] >= '2019-01-01') & (df['DT_NOTIFIC'] <= '2021-12-31') & (df['ANTIVIRAL'] == "Sim") & (df['DT_ANTIVIR'].isna()) & (df['TP_ANTIVIR'] != 'Em branco'), # 1 em branco/ignorado
                                                (df['DT_NOTIFIC'] >= '2019-01-01') & (df['DT_NOTIFIC'] <= '2021-12-31') & (df['ANTIVIRAL'] == "Sim") & (df['DT_ANTIVIR'].notna()) & (df['TP_ANTIVIR'] == 'Em branco'), # 1 em branco/ignorado
                                                (df['DT_NOTIFIC'] >= '2019-01-01') & (df['DT_NOTIFIC'] <= '2021-12-31') & (df['ANTIVIRAL'] == "Sim") & (df['DT_ANTIVIR'].isna()) & (df['TP_ANTIVIR'] == 'Em branco')], # 2 em branco/ignorado

                                               # Valores
                                               [0,
                                                1, 0, 1, 0,
                                                1, 0, 0, 1, 1, 2])
# DENOMINADOR
df['COMPL_HOSP_LAB_ANTIVIRAL_DEN'] = np.select([# 2009-2012
                                                (df['DT_NOTIFIC'] >= '2009-01-01') & (df['DT_NOTIFIC'] <= '2012-08-31'), # 0 campo - Campo não utilizado

                                                # 2013-2018
                                                (df['DT_NOTIFIC'] >= '2012-09-01') & (df['DT_NOTIFIC'] <= '2018-12-31') & (df['ANTIVIRAL'].isin(['Em branco','Ignorado'])), # 1 campo
                                                (df['DT_NOTIFIC'] >= '2012-09-01') & (df['DT_NOTIFIC'] <= '2018-12-31') & (df['ANTIVIRAL'] == "Não"), # 1 campo
                                                (df['DT_NOTIFIC'] >= '2012-09-01') & (df['DT_NOTIFIC'] <= '2018-12-31') & (df['ANTIVIRAL'] == "Sim"), # 2 campos

                                                # 2019-2021
                                                (df['DT_NOTIFIC'] >= '2019-01-01') & (df['DT_NOTIFIC'] <= '2021-12-31') & (df['ANTIVIRAL'].isin(['Em branco','Ignorado'])), # 1 campo
                                                (df['DT_NOTIFIC'] >= '2019-01-01') & (df['DT_NOTIFIC'] <= '2021-12-31') & (df['ANTIVIRAL'] == "Não"), # 1 campo
                                                (df['DT_NOTIFIC'] >= '2019-01-01') & (df['DT_NOTIFIC'] <= '2021-12-31') & (df['ANTIVIRAL'] == "Sim")], # 3 campos

                                               # Valores
                                               [0,
                                                1, 1, 2,
                                                1, 1, 3])


# A completitude da variável sobre uso do suporte ventilatório
# NÚMERADOR (CONTAR CAMPOS NULOS/IGNORADOS/EM BRANCO)
df['COMPL_HOSP_LAB_SUPORT_VEN_NUM'] = np.select([# 2009-2012
                                                (df['DT_NOTIFIC'] >= '2009-01-01') & (df['DT_NOTIFIC'] <= '2012-08-31'), # 0 em branco/ignorado - Campo não utilizado

                                                # 2013-2021
                                                (df['DT_NOTIFIC'] >= '2012-09-01') & (df['DT_NOTIFIC'] <= '2021-12-31') & (df['SUPORT_VEN'].isin(['Em branco','Ignorado'])), # 1 em branco/ignorado
                                                (df['DT_NOTIFIC'] >= '2012-09-01') & (df['DT_NOTIFIC'] <= '2021-12-31') & (df['SUPORT_VEN'].isin(['Sim, não invasivo','Sim, invasivo','Não']))], # 0 em branco/ignorado

                                               # Valores
                                               [0,
                                                1, 0])
# DENOMINADOR
df['COMPL_HOSP_LAB_SUPORT_VEN_DEN'] = np.select([# 2009-2012
                                                 (df['DT_NOTIFIC'] >= '2009-01-01') & (df['DT_NOTIFIC'] <= '2012-08-31'), # 0 campo - Campo não utilizado

                                                 # 2013-2018
                                                 (df['DT_NOTIFIC'] >= '2012-09-01') & (df['DT_NOTIFIC'] <= '2021-12-31')], # 1 campo

                                                # Valores
                                                [0,
                                                 1])


# A completitude das 3 variáveis sobre UTI ('UTI', 'DT_ENTUTI' e 'DT_SAIDUTI) será avaliada caso apresente uso da UTI ('UTI' == 'Sim')
# caso contrário será avaliada apenas a variável 'UTI'
# NÚMERADOR (CONTAR CAMPOS NULOS/IGNORADOS/EM BRANCO)
df['COMPL_HOSP_LAB_UTI_NUM'] = np.select([# 2009-2012
                                          (df['DT_NOTIFIC'] >= '2009-01-01') & (df['DT_NOTIFIC'] <= '2012-08-31'), # 0 em branco/ignorado - Campo não utilizado

                                          # 2013-2021
                                          (df['DT_NOTIFIC'] >= '2012-09-01') & (df['DT_NOTIFIC'] <= '2021-12-31') & (df['UTI'].isin(['Em branco','Ignorado'])), # 1 em branco/ignorado
                                          (df['DT_NOTIFIC'] >= '2012-09-01') & (df['DT_NOTIFIC'] <= '2021-12-31') & (df['UTI'] == "Não"), # 0 em branco/ignorado
                                          (df['DT_NOTIFIC'] >= '2012-09-01') & (df['DT_NOTIFIC'] <= '2021-12-31') & (df['UTI'] == "Sim") & (df['DT_ENTUTI'].notna()) & (df['DT_SAIDUTI'].notna()), # 0 em branco/ignorado
                                          (df['DT_NOTIFIC'] >= '2012-09-01') & (df['DT_NOTIFIC'] <= '2021-12-31') & (df['UTI'] == "Sim") & (df['DT_ENTUTI'].notna()) & (df['DT_SAIDUTI'].isna()), # 1 em branco/ignorado
                                          (df['DT_NOTIFIC'] >= '2012-09-01') & (df['DT_NOTIFIC'] <= '2021-12-31') & (df['UTI'] == "Sim") & (df['DT_ENTUTI'].isna()) & (df['DT_SAIDUTI'].notna()), # 1 em branco/ignorado
                                          (df['DT_NOTIFIC'] >= '2012-09-01') & (df['DT_NOTIFIC'] <= '2021-12-31') & (df['UTI'] == "Sim") & (df['DT_ENTUTI'].isna()) & (df['DT_SAIDUTI'].isna())], # 2 em branco/ignorado,

                                         # Valores
                                         [0,
                                          1, 0, 0, 1, 1, 2])
# DENOMINADOR
df['COMPL_HOSP_LAB_UTI_DEN'] = np.select([# 2009-2012
                                          (df['DT_NOTIFIC'] >= '2009-01-01') & (df['DT_NOTIFIC'] <= '2012-08-31'), # 0 em branco/ignorado - Campo não utilizado

                                          # 2013-2021
                                          (df['DT_NOTIFIC'] >= '2012-09-01') & (df['DT_NOTIFIC'] <= '2021-12-31') & (df['UTI'].isin(['Em branco','Ignorado'])), # 1 campo
                                          (df['DT_NOTIFIC'] >= '2012-09-01') & (df['DT_NOTIFIC'] <= '2021-12-31') & (df['UTI'] == "Não"), # 1 campo
                                          (df['DT_NOTIFIC'] >= '2012-09-01') & (df['DT_NOTIFIC'] <= '2021-12-31') & (df['UTI'] == "Sim")], # 3 campos

                                         # Valores
                                         [0,
                                          1, 1, 3])

# A completitude das 3 variáveis sobre exame PCR ('PCR_REALIZADO', 'DT_COLETA' e 'DT_PCR) será avaliada caso realizado o PCR ('PCR_REALIZADO' == 'Sim')
# caso contrário será avaliada apenas a variável 'HOSPITAL'
# NÚMERADOR (CONTAR CAMPOS NULOS/IGNORADOS/EM BRANCO)
df['COMPL_HOSP_LAB_PCR_NUM'] = np.select([# 2009-2012
                                          (df['DT_NOTIFIC'] >= '2009-01-01') & (df['DT_NOTIFIC'] <= '2012-08-31') & (df['PCR_REALIZADO'].isin(['Em branco','Ignorado'])), # 1 em branco/ignorado

                                          # 2013-2021
                                          (df['DT_NOTIFIC'] >= '2012-09-01') & (df['DT_NOTIFIC'] <= '2021-12-31') & (df['PCR_REALIZADO'].isin(['Em branco','Ignorado'])), # 1 em branco/ignorado
                                          (df['DT_NOTIFIC'] >= '2012-09-01') & (df['DT_NOTIFIC'] <= '2021-12-31') & (df['PCR_REALIZADO'] == "Não"), # 0 em branco/ignorado
                                          (df['DT_NOTIFIC'] >= '2012-09-01') & (df['DT_NOTIFIC'] <= '2021-12-31') & (df['PCR_REALIZADO'] == "Sim") & (df['DT_COLETA'].notna()) & (df['DT_PCR'].notna()), # 0 em branco/ignorado
                                          (df['DT_NOTIFIC'] >= '2012-09-01') & (df['DT_NOTIFIC'] <= '2021-12-31') & (df['PCR_REALIZADO'] == "Sim") & (df['DT_COLETA'].notna()) & (df['DT_PCR'].isna()), # 1 em branco/ignorado
                                          (df['DT_NOTIFIC'] >= '2012-09-01') & (df['DT_NOTIFIC'] <= '2021-12-31') & (df['PCR_REALIZADO'] == "Sim") & (df['DT_COLETA'].isna()) & (df['DT_PCR'].notna()), # 1 em branco/ignorado
                                          (df['DT_NOTIFIC'] >= '2012-09-01') & (df['DT_NOTIFIC'] <= '2021-12-31') & (df['PCR_REALIZADO'] == "Sim") & (df['DT_COLETA'].isna()) & (df['DT_PCR'].isna())], # 2 em branco/ignorado,

                                         # Valores
                                         [1,
                                          1, 0, 0, 1, 1, 2])
# DENOMINADOR
df['COMPL_HOSP_LAB_PCR_DEN'] = np.select([# 2009-2012
                                          (df['DT_NOTIFIC'] >= '2009-01-01') & (df['DT_NOTIFIC'] <= '2012-08-31'), # 1 campo

                                          # 2013-2021
                                          (df['DT_NOTIFIC'] >= '2012-09-01') & (df['DT_NOTIFIC'] <= '2021-12-31') & (df['PCR_REALIZADO'].isin(['Em branco','Ignorado'])), # 1 campo
                                          (df['DT_NOTIFIC'] >= '2012-09-01') & (df['DT_NOTIFIC'] <= '2021-12-31') & (df['PCR_REALIZADO'] == "Não"), # 1 campo
                                          (df['DT_NOTIFIC'] >= '2012-09-01') & (df['DT_NOTIFIC'] <= '2021-12-31') & (df['PCR_REALIZADO'] == "Sim")], # 3 campos

                                         # Valores
                                         [1,
                                          1, 1, 3])


# CALCULAR COMPLETITUDE - VARIÁVEIS HOSPITALARES E LABORATORIAIS
# Avaliar completitude das variáveis hospitalares e laboratorias, se percentual de completitude >70% (Boa) se <70% (Ruim)
df['COMPL_HOSP_LAB'] = (round(100-( (df['COMPL_HOSP_LAB_HOSPITAL_NUM'] + df['COMPL_HOSP_LAB_RAIOX_NUM'] + df['COMPL_HOSP_LAB_ANTIVIRAL_NUM'] + df['COMPL_HOSP_LAB_SUPORT_VEN_NUM'] + df['COMPL_HOSP_LAB_UTI_NUM'] + df['COMPL_HOSP_LAB_PCR_NUM']) / (df['COMPL_HOSP_LAB_HOSPITAL_DEN'] + df['COMPL_HOSP_LAB_RAIOX_DEN'] + df['COMPL_HOSP_LAB_ANTIVIRAL_DEN'] + df['COMPL_HOSP_LAB_SUPORT_VEN_DEN'] + df['COMPL_HOSP_LAB_UTI_DEN'] + df['COMPL_HOSP_LAB_PCR_DEN']) * 100), 2) > VALOR_CORTE).replace({True:1,False:0}).astype('category')

In [20]:
########### COMPL_INVEST_EPID ###########
########### COMPL_INVEST_EPID ###########
########### COMPL_INVEST_EPID ###########
# Avaliar completitude das variáveis sobre investigação epidemiológica, se percentual de completitude >70% (Boa) se <70% (Ruim)

# Versão 1 (4 variáveis sobre investigação epidemiológica)
df.loc[(df['DT_NOTIFIC'] >= '2009-01-01') & (df['DT_NOTIFIC'] <= '2018-12-31'),'COMPL_INVEST_EPID'] = (round(100-((df[['CLASSI_FIN','CRITERIO','EVOLUCAO','DT_ENCERRA']].isna().sum(axis=1) + # Testar se variáveis sobre datas são nulas
                                                                                                                   df[['CLASSI_FIN','CRITERIO','EVOLUCAO','DT_ENCERRA']].eq('Ignorado').sum(axis=1) + # Testar se variáveis categóricas são "Ignorado"
                                                                                                                   df[['CLASSI_FIN','CRITERIO','EVOLUCAO','DT_ENCERRA']].eq('Em branco').sum(axis=1)) # Testar se variáveis categóricas são "Em branco"
                                                                                                                   / 4) * 100, # Calcular percentual de completitude
                                                                                                             2) > VALOR_CORTE).replace({True:1,False:0}).astype('category') # Testar se percentual de completitude é >70% ou <70%

# Versão 2 (5 variáveis sobre investigação epidemiológica)
df.loc[(df['DT_NOTIFIC'] >= '2019-01-01') & (df['DT_NOTIFIC'] <= '2021-12-31'),'COMPL_INVEST_EPID'] = (round(100-((df[['CLASSI_FIN','CRITERIO','EVOLUCAO','DT_EVOLUCA','DT_ENCERRA']].isna().sum(axis=1) + # Testar se variáveis sobre datas são nulas
                                                                                                                   df[['CLASSI_FIN','CRITERIO','EVOLUCAO','DT_EVOLUCA','DT_ENCERRA']].eq('Ignorado').sum(axis=1) + # Testar se variáveis categóricas são "Ignorado"
                                                                                                                   df[['CLASSI_FIN','CRITERIO','EVOLUCAO','DT_EVOLUCA','DT_ENCERRA']].eq('Em branco').sum(axis=1)) # Testar se variáveis categóricas são "Em branco"
                                                                                                                   / 5) * 100, # Calcular percentual de completitude
                                                                                                             2) > VALOR_CORTE).replace({True:1,False:0}).astype('category') # Testar se percentual de completitude é >70% ou <70%

In [21]:
########### COMPL_TOTAL ###########
########### COMPL_TOTAL ###########
########### COMPL_TOTAL ###########

# Versão 1 (2009-2011)
df.loc[(df['DT_NOTIFIC'] >= '2009-01-01') & (df['DT_NOTIFIC'] <= '2012-08-31'),'COMPL_TOTAL_NUM'] = (df[["DT_NASC","CS_SEXO","CS_GESTANT","CS_RACA","CS_ESCOL_N","VACINA","CO_MUN_RES", # 7 variáveis
                                                                                                         "DT_SIN_PRI","FEBRE","TOSSE","DISPNEIA","GARGANTA","MIALGIA","CALAFRIO","ARTRALGIA","CONJUNTIV","CORIZA","DIARREIA", # 11 variáveis
                                                                                                         'CARDIOPATI','PNEUMOPATI','RENAL','IMUNODEPRE','HEMOGLOBI','TABAGISMO','METABOLICA', # 7 variáveis
                                                                                                         'CLASSI_FIN','CRITERIO','EVOLUCAO','DT_ENCERRA']].isna().sum(axis=1) + # 4 variáveis
                                                                                                     df[["DT_NASC","CS_SEXO","CS_GESTANT","CS_RACA","CS_ESCOL_N","VACINA","CO_MUN_RES",
                                                                                                         "DT_SIN_PRI","FEBRE","TOSSE","DISPNEIA","GARGANTA","MIALGIA","CALAFRIO","ARTRALGIA","CONJUNTIV","CORIZA","DIARREIA",
                                                                                                         'CARDIOPATI','PNEUMOPATI','RENAL','IMUNODEPRE','HEMOGLOBI','TABAGISMO','METABOLICA',
                                                                                                         'CLASSI_FIN','CRITERIO','EVOLUCAO','DT_ENCERRA']].eq('Ignorado').sum(axis=1) +
                                                                                                     df[["DT_NASC","CS_SEXO","CS_GESTANT","CS_RACA","CS_ESCOL_N","VACINA","CO_MUN_RES",
                                                                                                         "DT_SIN_PRI","FEBRE","TOSSE","DISPNEIA","GARGANTA","MIALGIA","CALAFRIO","ARTRALGIA","CONJUNTIV","CORIZA","DIARREIA",
                                                                                                         'CARDIOPATI','PNEUMOPATI','RENAL','IMUNODEPRE','HEMOGLOBI','TABAGISMO','METABOLICA',
                                                                                                         'CLASSI_FIN','CRITERIO','EVOLUCAO','DT_ENCERRA']].eq('Em branco').sum(axis=1))
df.loc[(df['DT_NOTIFIC'] >= '2009-01-01') & (df['DT_NOTIFIC'] <= '2012-08-31'),'COMPL_TOTAL_DEN'] = 7 + 11 + 7 + 4





# Versão 2 (2013-2018)
df.loc[(df['DT_NOTIFIC'] >= '2012-09-01') & (df['DT_NOTIFIC'] <= '2018-12-31'),'COMPL_TOTAL_NUM'] = (df[["DT_NASC","CS_SEXO","CS_GESTANT","CS_RACA","CS_ESCOL_N","VACINA","CO_MUN_RES", # 7 variáveis
                                                                                                         "DT_SIN_PRI","FEBRE","TOSSE","DISPNEIA","GARGANTA","MIALGIA","DESC_RESP","SATURACAO", # 8 variáveis
                                                                                                         'CARDIOPATI','PNEUMOPATI','RENAL','IMUNODEPRE','HEPATICA','NEUROLOGIC','SIND_DOWN','PUERPERA','OBESIDADE', # 9 variáveis
                                                                                                         'CLASSI_FIN','CRITERIO','EVOLUCAO','DT_ENCERRA']].isna().sum(axis=1) + # 4 variáveis
                                                                                                     df[["DT_NASC","CS_SEXO","CS_GESTANT","CS_RACA","CS_ESCOL_N","VACINA","CO_MUN_RES",
                                                                                                         "DT_SIN_PRI","FEBRE","TOSSE","DISPNEIA","GARGANTA","MIALGIA","DESC_RESP","SATURACAO",
                                                                                                         'CARDIOPATI','PNEUMOPATI','RENAL','IMUNODEPRE','HEPATICA','NEUROLOGIC','SIND_DOWN','PUERPERA','OBESIDADE',
                                                                                                         'CLASSI_FIN','CRITERIO','EVOLUCAO','DT_ENCERRA']].eq('Ignorado').sum(axis=1) +
                                                                                                     df[["DT_NASC","CS_SEXO","CS_GESTANT","CS_RACA","CS_ESCOL_N","VACINA","CO_MUN_RES",
                                                                                                         "DT_SIN_PRI","FEBRE","TOSSE","DISPNEIA","GARGANTA","MIALGIA","DESC_RESP","SATURACAO",
                                                                                                         'CARDIOPATI','PNEUMOPATI','RENAL','IMUNODEPRE','HEPATICA','NEUROLOGIC','SIND_DOWN','PUERPERA','OBESIDADE',
                                                                                                         'CLASSI_FIN','CRITERIO','EVOLUCAO','DT_ENCERRA']].eq('Em branco').sum(axis=1))
df.loc[(df['DT_NOTIFIC'] >= '2012-09-01') & (df['DT_NOTIFIC'] <= '2018-12-31'),'COMPL_TOTAL_DEN'] = 7 + 8 + 9 + 4





# Versão 3 (2019-2020)
df.loc[(df['DT_NOTIFIC'] >= '2019-01-01') & (df['DT_NOTIFIC'] <= '2020-07-31'),'COMPL_TOTAL_NUM'] = (df[["DT_NASC","CS_SEXO","CS_GESTANT","CS_RACA","CS_ESCOL_N",'CS_ZONA',"VACINA","CO_MUN_RES", # 8 variáveis
                                                                                                         "DT_SIN_PRI","FEBRE","TOSSE","DISPNEIA","GARGANTA","DIARREIA","DESC_RESP","SATURACAO","VOMITO", # 9 variáveis
                                                                                                         'CARDIOPATI','PNEUMOPATI','RENAL','IMUNODEPRE','HEPATICA','NEUROLOGIC','SIND_DOWN','PUERPERA','OBESIDADE','HEMATOLOGI','ASMA','DIABETES', # 12 variáveis
                                                                                                         'CLASSI_FIN','CRITERIO','EVOLUCAO','DT_EVOLUCA','DT_ENCERRA']].isna().sum(axis=1) + # 5 variáveis
                                                                                                     df[["DT_NASC","CS_SEXO","CS_GESTANT","CS_RACA","CS_ESCOL_N",'CS_ZONA',"VACINA","CO_MUN_RES",
                                                                                                         "DT_SIN_PRI","FEBRE","TOSSE","DISPNEIA","GARGANTA","DIARREIA","DESC_RESP","SATURACAO","VOMITO",
                                                                                                         'CARDIOPATI','PNEUMOPATI','RENAL','IMUNODEPRE','HEPATICA','NEUROLOGIC','SIND_DOWN','PUERPERA','OBESIDADE','HEMATOLOGI','ASMA','DIABETES',
                                                                                                         'CLASSI_FIN','CRITERIO','EVOLUCAO','DT_EVOLUCA','DT_ENCERRA']].eq('Ignorado').sum(axis=1) +
                                                                                                     df[["DT_NASC","CS_SEXO","CS_GESTANT","CS_RACA","CS_ESCOL_N",'CS_ZONA',"VACINA","CO_MUN_RES",
                                                                                                         "DT_SIN_PRI","FEBRE","TOSSE","DISPNEIA","GARGANTA","DIARREIA","DESC_RESP","SATURACAO","VOMITO",
                                                                                                         'CARDIOPATI','PNEUMOPATI','RENAL','IMUNODEPRE','HEPATICA','NEUROLOGIC','SIND_DOWN','PUERPERA','OBESIDADE','HEMATOLOGI','ASMA','DIABETES',
                                                                                                         'CLASSI_FIN','CRITERIO','EVOLUCAO','DT_EVOLUCA','DT_ENCERRA']].eq('Em branco').sum(axis=1))
df.loc[(df['DT_NOTIFIC'] >= '2019-01-01') & (df['DT_NOTIFIC'] <= '2020-07-31'),'COMPL_TOTAL_DEN'] = 8 + 9 + 12 + 5





# Versão 4 (2020-2021)
df.loc[(df['DT_NOTIFIC'] >= '2020-08-01') & (df['DT_NOTIFIC'] <= '2021-12-31'),'COMPL_TOTAL_NUM'] = (df[["DT_NASC","CS_SEXO","CS_GESTANT","CS_RACA","CS_ESCOL_N",'CS_ZONA',"VACINA","HISTO_VGM","CO_MUN_RES", # 9 variáveis
                                                                                                         "DT_SIN_PRI","FEBRE","TOSSE","DISPNEIA","GARGANTA","DIARREIA","DESC_RESP","SATURACAO","VOMITO","DOR_ABD","FADIGA","PERD_OLFT","PERD_PALA", # 13 variáveis
                                                                                                         'CARDIOPATI','PNEUMOPATI','RENAL','IMUNODEPRE','HEPATICA','NEUROLOGIC','SIND_DOWN','PUERPERA','OBESIDADE','HEMATOLOGI','ASMA','DIABETES', # 12 variáveis
                                                                                                         'CLASSI_FIN','CRITERIO','EVOLUCAO','DT_EVOLUCA','DT_ENCERRA']].isna().sum(axis=1) + # 5 variáveis
                                                                                                     df[["DT_NASC","CS_SEXO","CS_GESTANT","CS_RACA","CS_ESCOL_N",'CS_ZONA',"VACINA","HISTO_VGM","CO_MUN_RES",
                                                                                                         "DT_SIN_PRI","FEBRE","TOSSE","DISPNEIA","GARGANTA","DIARREIA","DESC_RESP","SATURACAO","VOMITO","DOR_ABD","FADIGA","PERD_OLFT","PERD_PALA",
                                                                                                         'CARDIOPATI','PNEUMOPATI','RENAL','IMUNODEPRE','HEPATICA','NEUROLOGIC','SIND_DOWN','PUERPERA','OBESIDADE','HEMATOLOGI','ASMA','DIABETES',
                                                                                                         'CLASSI_FIN','CRITERIO','EVOLUCAO','DT_EVOLUCA','DT_ENCERRA']].eq('Ignorado').sum(axis=1) +
                                                                                                     df[["DT_NASC","CS_SEXO","CS_GESTANT","CS_RACA","CS_ESCOL_N",'CS_ZONA',"VACINA","HISTO_VGM","CO_MUN_RES",
                                                                                                         "DT_SIN_PRI","FEBRE","TOSSE","DISPNEIA","GARGANTA","DIARREIA","DESC_RESP","SATURACAO","VOMITO","DOR_ABD","FADIGA","PERD_OLFT","PERD_PALA",
                                                                                                         'CARDIOPATI','PNEUMOPATI','RENAL','IMUNODEPRE','HEPATICA','NEUROLOGIC','SIND_DOWN','PUERPERA','OBESIDADE','HEMATOLOGI','ASMA','DIABETES',
                                                                                                         'CLASSI_FIN','CRITERIO','EVOLUCAO','DT_EVOLUCA','DT_ENCERRA']].eq('Em branco').sum(axis=1))
df.loc[(df['DT_NOTIFIC'] >= '2020-08-01') & (df['DT_NOTIFIC'] <= '2021-12-31'),'COMPL_TOTAL_DEN'] = 9 + 13 + 12 + 5


# CALCULAR COMPLETITUDE - TODAS VARIÁVEIS
# Avaliar completitude das variáveis, se percentual de completitude >70% (Boa) se <70% (Ruim)
df['COMPL_TOTAL_NUM'] = df['COMPL_TOTAL_NUM'] + df[['COMPL_HOSP_LAB_HOSPITAL_NUM','COMPL_HOSP_LAB_RAIOX_NUM','COMPL_HOSP_LAB_ANTIVIRAL_NUM','COMPL_HOSP_LAB_SUPORT_VEN_NUM','COMPL_HOSP_LAB_UTI_NUM','COMPL_HOSP_LAB_PCR_NUM']].sum(axis=1)
df['COMPL_TOTAL_DEN'] = df['COMPL_TOTAL_DEN'] + df[['COMPL_HOSP_LAB_HOSPITAL_DEN','COMPL_HOSP_LAB_RAIOX_DEN','COMPL_HOSP_LAB_ANTIVIRAL_DEN','COMPL_HOSP_LAB_SUPORT_VEN_DEN','COMPL_HOSP_LAB_UTI_DEN','COMPL_HOSP_LAB_PCR_DEN']].sum(axis=1)

df['COMPL_TOTAL'] = (round(100-(df['COMPL_TOTAL_NUM'] / df['COMPL_TOTAL_DEN']) * 100, # Calcular percentual de completitude
                                2) > VALOR_CORTE).replace({True:1,False:0}).astype('category') # Testar se percentual de completitude é >70% ou <70%


# Remover variáveis não utilizadas
df.drop(['COMPL_HOSP_LAB_HOSPITAL_NUM','COMPL_HOSP_LAB_RAIOX_NUM','COMPL_HOSP_LAB_ANTIVIRAL_NUM','COMPL_HOSP_LAB_SUPORT_VEN_NUM','COMPL_HOSP_LAB_UTI_NUM','COMPL_HOSP_LAB_PCR_NUM','COMPL_TOTAL_NUM',
         'COMPL_HOSP_LAB_HOSPITAL_DEN','COMPL_HOSP_LAB_RAIOX_DEN','COMPL_HOSP_LAB_ANTIVIRAL_DEN','COMPL_HOSP_LAB_SUPORT_VEN_DEN','COMPL_HOSP_LAB_UTI_DEN','COMPL_HOSP_LAB_PCR_DEN','COMPL_TOTAL_DEN'], axis=1, inplace=True)

## 4.2 Oportunidade

In [22]:
########### OPORT_IDENT ###########
########### OPORT_IDENT ###########
########### OPORT_IDENT ###########
# Oportunidade de identificação
df['OPORT_IDENT'] = (df['DT_NOTIFIC'] - df['DT_SIN_PRI']).dt.days.astype('Int64') # Calcular tempo entre datas e converter para formato numérico
df.loc[(df['DT_NOTIFIC'].isna()) | (df['DT_SIN_PRI'].isna()), 'OPORT_IDENT'] = np.nan # Caso apresente campos de data em branco substituir oportunidade para nulo
df.loc[df['OPORT_IDENT'].notna(), 'OPORT_IDENT'] = np.where(df[df['OPORT_IDENT'].notna()]['OPORT_IDENT'].between(0,1), 1, 0) # Classificar em oportuno (1) e inoportuno (0)

########### OPORT_NOTIFC ###########
########### OPORT_NOTIFC ###########
########### OPORT_NOTIFC ###########
# Oportunidade de notificação
df['OPORT_NOTIFC'] = (df['DT_NOTIFIC'] - df['DT_INTERNA']).dt.days.astype('Int64') # Calcular tempo entre datas e converter para formato numérico
df.loc[(df['DT_NOTIFIC'].isna()) | (df['DT_INTERNA'].isna()), 'OPORT_NOTIFC'] = np.nan # Caso apresente campos de data em branco substituir oportunidade para nulo
df.loc[df['OPORT_NOTIFC'].notna(), 'OPORT_NOTIFC'] = np.where(df[df['OPORT_NOTIFC'].notna()]['OPORT_NOTIFC'].between(0,1), 1, 0) # Classificar em oportuno (1) e inoportuno (0)

########### OPORT_DIGITA ###########
########### OPORT_DIGITA ###########
########### OPORT_DIGITA ###########
# Oportunidade de digitação
df['OPORT_DIGITA'] = (df['DT_DIGITA'] - df['DT_NOTIFIC']).dt.days.astype('Int64') # Calcular tempo entre datas e converter para formato numérico
df.loc[(df['DT_DIGITA'].isna()) | (df['DT_NOTIFIC'].isna()), 'OPORT_DIGITA'] = np.nan # Caso apresente campos de data em branco substituir oportunidade para nulo
df.loc[df['OPORT_DIGITA'].notna(), 'OPORT_DIGITA'] = np.where(df[df['OPORT_DIGITA'].notna()]['OPORT_DIGITA'].between(0,1), 1, 0) # Classificar em oportuno (1) e inoportuno (0)

########### OPORT_COL ###########
########### OPORT_COL ###########
########### OPORT_COL ###########
# Oportunidade de coleta
df['OPORT_COL'] = (df['DT_COLETA'] - df['DT_SIN_PRI']).dt.days.astype('Int64') # Calcular tempo entre datas e converter para formato numérico
df.loc[(df['DT_COLETA'].isna()) | (df['DT_SIN_PRI'].isna()), 'OPORT_COL'] = np.nan # Caso apresente campos de data em branco substituir oportunidade para nulo
df.loc[df['OPORT_COL'].notna(), 'OPORT_COL'] = np.where(df[df['OPORT_COL'].notna()]['OPORT_COL'].between(3,5), 1, 0) # Classificar em oportuno (1) e inoportuno (0)

########### OPORT_RES ###########
########### OPORT_RES ###########
########### OPORT_RES ###########
# Oportunidade de resultado
df['OPORT_RES'] = (df['DT_PCR'] - df['DT_COLETA']).dt.days.astype('Int64') # Calcular tempo entre datas e converter para formato numérico
df.loc[(df['DT_PCR'].isna()) | (df['DT_COLETA'].isna()), 'OPORT_RES'] = np.nan # Caso apresente campos de data em branco substituir oportunidade para nulo
df.loc[df['OPORT_RES'].notna(), 'OPORT_RES'] = np.where(df[df['OPORT_RES'].notna()]['OPORT_RES'].between(0,7), 1, 0) # Classificar em oportuno (1) e inoportuno (0)

########### OPORT_ENC ###########
########### OPORT_ENC ###########
########### OPORT_ENC ###########
# Oportunidade de encerração
df['OPORT_ENC'] = (df['DT_ENCERRA'] - df['DT_NOTIFIC']).dt.days.astype('Int64') # Calcular tempo entre datas e converter para formato numérico
df.loc[(df['DT_ENCERRA'].isna()) | (df['DT_NOTIFIC'].isna()), 'OPORT_ENC'] = np.nan # Caso apresente campos de data em branco substituir oportunidade para nulo
df.loc[df['OPORT_ENC'].notna(), 'OPORT_ENC'] = np.where(df[df['OPORT_ENC'].notna()]['OPORT_ENC'].between(0,60), 1, 0) # Classificar em oportuno (1) e inoportuno (0)

<ipython-input-22-5c13c42ecc8d>:7: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  df.loc[df['OPORT_IDENT'].notna(), 'OPORT_IDENT'] = np.where(df[df['OPORT_IDENT'].notna()]['OPORT_IDENT'].between(0,1), 1, 0) # Classificar em oportuno (1) e inoportuno (0)


# 5 Figures and tables


In [23]:
df.head(1)

,DT_NOTIFIC,NU_ANO,NU_MES,DT_DIGITA,PANDEMIA,CO_MUN_NOT,SG_UF_NOT,REGIAO_NOT,PORTE,FRONTEIRA,CO_MUN_RES,DT_NASC,NU_IDADE_N,GRUPO_IDADE,CS_SEXO,CS_GESTANT,CS_RACA,CS_ESCOL_N,CS_ZONA,VACINA,HISTO_VGM,DT_SIN_PRI,FEBRE,TOSSE,CALAFRIO,DISPNEIA,GARGANTA,ARTRALGIA,MIALGIA,CONJUNTIV,CORIZA,DIARREIA,DOR_ABD,FADIGA,PERD_OLFT,PERD_PALA,VOMITO,DESC_RESP,SATURACAO,CARDIOPATI,PNEUMOPATI,RENAL,HEMOGLOBI,IMUNODEPRE,TABAGISMO,METABOLICA,HEPATICA,NEUROLOGIC,OBESIDADE,PUERPERA,SIND_DOWN,HEMATOLOGI,ASMA,DIABETES,HOSPITAL,DT_INTERNA,CO_MUN_INT,RAIOX_RES,DT_RAIOX,ANTIVIRAL,DT_ANTIVIR,TP_ANTIVIR,UTI,DT_ENTUTI,DT_SAIDUTI,SUPORT_VEN,PCR_REALIZADO,DT_COLETA,DT_PCR,CLASSI_FIN,CRITERIO,EVOLUCAO,DT_EVOLUCA,DT_ENCERRA,COMPL_IDENT,COMPL_SINTO,COMPL_RISCO,COMPL_HOSP_LAB,COMPL_INVEST_EPID,COMPL_TOTAL,OPORT_IDENT,OPORT_NOTIFC,OPORT_DIGITA,OPORT_COL,OPORT_RES,OPORT_ENC
167914,2009-04-24,2009,4,2009-12-16,Anterior,355030,SP,Sudeste,Grande,Sim,355030,1983-11-30,25.0,0 a 29 anos,Feminino,Não,Branca,Ignorado,Em branco,Ignorado,Em branco,2009-04-24,Sim,Não,Não,Não,Não,Não,Sim,Não,Sim,Não,Em branco,Em branco,Em branco,Em branco,Em branco,Em branco,Em branco,Não,Não,Não,Não,Não,Não,Não,Em branco,Em branco,Em branco,Em branco,Em branco,Em branco,Em branco,Em branco,Não,NaT,Em branco,Em branco,NaT,Em branco,NaT,Em branco,Em branco,NaT,NaT,Em branco,Sim,NaT,NaT,Não especificado,Laboratorial,Ignorado,NaT,2010-09-16,1,1,1,0,1,1,1,<NA>,0,<NA>,<NA>,0


In [24]:
# Table 1
consulta = pd.DataFrame(pd.pivot_table(df,
                                       values='DT_NOTIFIC',
                                       index=['REGIAO_NOT'],
                                       columns=['OPORT_RES'],
                                       aggfunc='count', fill_value=0, dropna=True, margins=True).to_records())
#consulta['%'] = round(consulta['DT_NOTIFIC'] / consulta['DT_NOTIFIC'].iloc[-1] * 100, 1)
consulta

,REGIAO_NOT,0,1,All
0,Centro-Oeste,16834,176986,193820
1,Nordeste,43846,322624,366470
2,Norte,10569,80921,91490
3,Sudeste,120071,1085122,1205193
4,Sul,37472,377713,415185
5,All,228792,2043366,2272158


In [ ]:
# Export data to Pearson's chi-squared test
df[['REGIAO_NOT','PORTE','FRONTEIRA',
    'COMPL_IDENT','COMPL_SINTO','COMPL_RISCO','COMPL_HOSP_LAB','COMPL_INVEST_EPID','COMPL_TOTAL',
    'OPORT_IDENT','OPORT_NOTIFC','OPORT_DIGITA','OPORT_COL','OPORT_RES','OPORT_ENC']].to_csv('dataset_chi_squared.csv')

# Export data to Logistic regression analysis
df[df['EVOLUCAO'].isin(['Cura','Óbito'])][['EVOLUCAO',
                                           'COMPL_IDENT','COMPL_SINTO','COMPL_RISCO','COMPL_HOSP_LAB','COMPL_INVEST_EPID','COMPL_TOTAL',
                                           'OPORT_IDENT','OPORT_NOTIFC','OPORT_DIGITA','OPORT_COL','OPORT_RES','OPORT_ENC']].dropna().to_csv('dataset_logistic_regression.csv')

In [ ]:
# CausalImpact analysis
index_ano_mes = pd.DataFrame(pd.pivot_table(df, values='DT_NOTIFIC', index=['NU_ANO','NU_MES'], aggfunc='count', fill_value=0, dropna=False, margins=True).to_records())
index_ano_mes['NU_ANO_MES'] = index_ano_mes['NU_ANO'].astype(str) + "-" + index_ano_mes['NU_MES'].astype(str) + "-1"
index_ano_mes = index_ano_mes[['NU_ANO_MES']][:-1]

percentual_compl_score_identificacao = pd.DataFrame(pd.pivot_table(df, values='DT_NOTIFIC', index=['NU_ANO','NU_MES'], columns=['COMPL_IDENT'], aggfunc='count', fill_value=0, dropna=False, margins=True).to_records())
percentual_compl_score_identificacao['percentual_compl_score_identificacao'] = percentual_compl_score_identificacao['1'] / percentual_compl_score_identificacao['All'] * 100
percentual_compl_score_identificacao['NU_ANO_MES'] = percentual_compl_score_identificacao['NU_ANO'].astype(str).str[0:4] + "-" + percentual_compl_score_identificacao['NU_MES'].astype(str) + "-1"
percentual_compl_score_identificacao = percentual_compl_score_identificacao[['NU_ANO_MES','percentual_compl_score_identificacao']]

percentual_compl_score_sintomas = pd.DataFrame(pd.pivot_table(df, values='DT_NOTIFIC', index=['NU_ANO','NU_MES'], columns=['COMPL_SINTO'], aggfunc='count', fill_value=0, dropna=False, margins=True).to_records())
percentual_compl_score_sintomas['percentual_compl_score_sintomas'] = percentual_compl_score_sintomas['1'] / percentual_compl_score_sintomas['All'] * 100
percentual_compl_score_sintomas['NU_ANO_MES'] = percentual_compl_score_sintomas['NU_ANO'].astype(str).str[0:4] + "-" + percentual_compl_score_sintomas['NU_MES'].astype(str) + "-1"
percentual_compl_score_sintomas = percentual_compl_score_sintomas[['NU_ANO_MES','percentual_compl_score_sintomas']]

percentual_compl_score_comorbidades = pd.DataFrame(pd.pivot_table(df, values='DT_NOTIFIC', index=['NU_ANO','NU_MES'], columns=['COMPL_RISCO'], aggfunc='count', fill_value=0, dropna=False, margins=True).to_records())
percentual_compl_score_comorbidades['percentual_compl_score_comorbidades'] = percentual_compl_score_comorbidades['1'] / percentual_compl_score_comorbidades['All'] * 100
percentual_compl_score_comorbidades['NU_ANO_MES'] = percentual_compl_score_comorbidades['NU_ANO'].astype(str).str[0:4] + "-" + percentual_compl_score_comorbidades['NU_MES'].astype(str) + "-1"
percentual_compl_score_comorbidades = percentual_compl_score_comorbidades[['NU_ANO_MES','percentual_compl_score_comorbidades']]

percentual_compl_score_hospitalar = pd.DataFrame(pd.pivot_table(df, values='DT_NOTIFIC', index=['NU_ANO','NU_MES'], columns=['COMPL_HOSP_LAB'], aggfunc='count', fill_value=0, dropna=False, margins=True).to_records())
percentual_compl_score_hospitalar['percentual_compl_score_hospitalar'] = percentual_compl_score_hospitalar['1'] / percentual_compl_score_hospitalar['All'] * 100
percentual_compl_score_hospitalar['NU_ANO_MES'] = percentual_compl_score_hospitalar['NU_ANO'].astype(str).str[0:4] + "-" + percentual_compl_score_hospitalar['NU_MES'].astype(str) + "-1"
percentual_compl_score_hospitalar = percentual_compl_score_hospitalar[['NU_ANO_MES','percentual_compl_score_hospitalar']]

percentual_compl_score_investigacao = pd.DataFrame(pd.pivot_table(df, values='DT_NOTIFIC', index=['NU_ANO','NU_MES'], columns=['COMPL_INVEST_EPID'], aggfunc='count', fill_value=0, dropna=False, margins=True).to_records())
percentual_compl_score_investigacao['percentual_compl_score_investigacao'] = percentual_compl_score_investigacao['1'] / percentual_compl_score_investigacao['All'] * 100
percentual_compl_score_investigacao['NU_ANO_MES'] = percentual_compl_score_investigacao['NU_ANO'].astype(str).str[0:4] + "-" + percentual_compl_score_investigacao['NU_MES'].astype(str) + "-1"
percentual_compl_score_investigacao = percentual_compl_score_investigacao[['NU_ANO_MES','percentual_compl_score_investigacao']]

percentual_compl_score_total = pd.DataFrame(pd.pivot_table(df, values='DT_NOTIFIC', index=['NU_ANO','NU_MES'], columns=['COMPL_TOTAL'], aggfunc='count', fill_value=0, dropna=False, margins=True).to_records())
percentual_compl_score_total['percentual_compl_score_total'] = percentual_compl_score_total['1'] / percentual_compl_score_total['All'] * 100
percentual_compl_score_total['NU_ANO_MES'] = percentual_compl_score_total['NU_ANO'].astype(str).str[0:4] + "-" + percentual_compl_score_total['NU_MES'].astype(str) + "-1"
percentual_compl_score_total = percentual_compl_score_total[['NU_ANO_MES','percentual_compl_score_total']]

percentual_oport_score_identificacao = pd.DataFrame(pd.pivot_table(df, values='DT_NOTIFIC', index=['NU_ANO','NU_MES'], columns=['OPORT_IDENT'], aggfunc='count', fill_value=0, dropna=False, margins=True).to_records())
percentual_oport_score_identificacao['percentual_oport_score_identificacao'] = percentual_oport_score_identificacao['1'] / percentual_oport_score_identificacao['All'] * 100
percentual_oport_score_identificacao['NU_ANO_MES'] = percentual_oport_score_identificacao['NU_ANO'].astype(str).str[0:4] + "-" + percentual_oport_score_identificacao['NU_MES'].astype(str) + "-1"
percentual_oport_score_identificacao = percentual_oport_score_identificacao[['NU_ANO_MES','percentual_oport_score_identificacao']]

percentual_oport_score_notificacao = pd.DataFrame(pd.pivot_table(df, values='DT_NOTIFIC', index=['NU_ANO','NU_MES'], columns=['OPORT_NOTIFC'], aggfunc='count', fill_value=0, dropna=False, margins=True).to_records())
percentual_oport_score_notificacao['percentual_oport_score_notificacao'] = percentual_oport_score_notificacao['1'] / percentual_oport_score_notificacao['All'] * 100
percentual_oport_score_notificacao['NU_ANO_MES'] = percentual_oport_score_notificacao['NU_ANO'].astype(str).str[0:4] + "-" + percentual_oport_score_notificacao['NU_MES'].astype(str) + "-1"
percentual_oport_score_notificacao = percentual_oport_score_notificacao[['NU_ANO_MES','percentual_oport_score_notificacao']]

percentual_oport_score_digitacao = pd.DataFrame(pd.pivot_table(df, values='DT_NOTIFIC', index=['NU_ANO','NU_MES'], columns=['OPORT_DIGITA'], aggfunc='count', fill_value=0, dropna=False, margins=True).to_records())
percentual_oport_score_digitacao['percentual_oport_score_digitacao'] = percentual_oport_score_digitacao['1'] / percentual_oport_score_digitacao['All'] * 100
percentual_oport_score_digitacao['NU_ANO_MES'] = percentual_oport_score_digitacao['NU_ANO'].astype(str).str[0:4] + "-" + percentual_oport_score_digitacao['NU_MES'].astype(str) + "-1"
percentual_oport_score_digitacao = percentual_oport_score_digitacao[['NU_ANO_MES','percentual_oport_score_digitacao']]

percentual_oport_score_coleta = pd.DataFrame(pd.pivot_table(df, values='DT_NOTIFIC', index=['NU_ANO','NU_MES'], columns=['OPORT_COL'], aggfunc='count', fill_value=0, dropna=False, margins=True).to_records())
percentual_oport_score_coleta['percentual_oport_score_coleta'] = percentual_oport_score_coleta['1'] / percentual_oport_score_coleta['All'] * 100
percentual_oport_score_coleta['NU_ANO_MES'] = percentual_oport_score_coleta['NU_ANO'].astype(str).str[0:4] + "-" + percentual_oport_score_coleta['NU_MES'].astype(str) + "-1"
percentual_oport_score_coleta = percentual_oport_score_coleta[['NU_ANO_MES','percentual_oport_score_coleta']]

percentual_oport_score_resultado = pd.DataFrame(pd.pivot_table(df, values='DT_NOTIFIC', index=['NU_ANO','NU_MES'], columns=['OPORT_RES'], aggfunc='count', fill_value=0, dropna=False, margins=True).to_records())
percentual_oport_score_resultado['percentual_oport_score_resultado'] = percentual_oport_score_resultado['1'] / percentual_oport_score_resultado['All'] * 100
percentual_oport_score_resultado['NU_ANO_MES'] = percentual_oport_score_resultado['NU_ANO'].astype(str).str[0:4] + "-" + percentual_oport_score_resultado['NU_MES'].astype(str) + "-1"
percentual_oport_score_resultado = percentual_oport_score_resultado[['NU_ANO_MES','percentual_oport_score_resultado']]

percentual_oport_score_investigacao = pd.DataFrame(pd.pivot_table(df, values='DT_NOTIFIC', index=['NU_ANO','NU_MES'], columns=['OPORT_ENC'], aggfunc='count', fill_value=0, dropna=False, margins=True).to_records())
percentual_oport_score_investigacao['percentual_oport_score_investigacao'] = percentual_oport_score_investigacao['1'] / percentual_oport_score_investigacao['All'] * 100
percentual_oport_score_investigacao['NU_ANO_MES'] = percentual_oport_score_investigacao['NU_ANO'].astype(str).str[0:4] + "-" + percentual_oport_score_investigacao['NU_MES'].astype(str) + "-1"
percentual_oport_score_investigacao = percentual_oport_score_investigacao[['NU_ANO_MES','percentual_oport_score_investigacao']]

pw_final = index_ano_mes.merge(percentual_compl_score_identificacao, left_on=['NU_ANO_MES'], right_on=['NU_ANO_MES'], how='left')
pw_final = pw_final.merge(percentual_compl_score_sintomas, left_on=['NU_ANO_MES'], right_on=['NU_ANO_MES'], how='left')
pw_final = pw_final.merge(percentual_compl_score_comorbidades, left_on=['NU_ANO_MES'], right_on=['NU_ANO_MES'], how='left')
pw_final = pw_final.merge(percentual_compl_score_hospitalar, left_on=['NU_ANO_MES'], right_on=['NU_ANO_MES'], how='left')
pw_final = pw_final.merge(percentual_compl_score_investigacao, left_on=['NU_ANO_MES'], right_on=['NU_ANO_MES'], how='left')
pw_final = pw_final.merge(percentual_compl_score_total, left_on=['NU_ANO_MES'], right_on=['NU_ANO_MES'], how='left')
pw_final = pw_final.merge(percentual_oport_score_identificacao, left_on=['NU_ANO_MES'], right_on=['NU_ANO_MES'], how='left')
pw_final = pw_final.merge(percentual_oport_score_notificacao, left_on=['NU_ANO_MES'], right_on=['NU_ANO_MES'], how='left')
pw_final = pw_final.merge(percentual_oport_score_digitacao, left_on=['NU_ANO_MES'], right_on=['NU_ANO_MES'], how='left')
pw_final = pw_final.merge(percentual_oport_score_coleta, left_on=['NU_ANO_MES'], right_on=['NU_ANO_MES'], how='left')
pw_final = pw_final.merge(percentual_oport_score_resultado, left_on=['NU_ANO_MES'], right_on=['NU_ANO_MES'], how='left')
pw_final = pw_final.merge(percentual_oport_score_investigacao, left_on=['NU_ANO_MES'], right_on=['NU_ANO_MES'], how='left')
pw_final = pw_final.reset_index()
pw_final.rename(columns={'NU_ANO_MES':'tempo','index':'ordem'}, inplace=True)
pw_final['tempo'] = pd.to_datetime(pw_final['tempo'], format='%Y/%m/%d', errors='coerce')
pw_final['pandemia'] = np.where(pw_final['tempo'] >= "2020-02-01","Durante","Antes") # 1 = Durante, 0 = Antes
pw_final['P'] = pw_final.groupby(['pandemia']).cumcount() # Index segundo pandemia
pw_final = pw_final[3:] # Filtrar primeiros 3 meses, todas variáveis com valores nulos

# CausalImpact analysis
!pip install pycausalimpact --q
from causalimpact import CausalImpact

pw_final['tempo'] = pd.to_datetime(pw_final['tempo'], format='%d/%m/%Y', errors='coerce')
pw_final = pw_final[['tempo','percentual_compl_score_identificacao']]
pw_final['percentual'] = pw_final['percentual_compl_score_identificacao'].astype(str).str.replace(",",".").astype(float)
pw_final = pw_final.set_index('tempo')

pre_period = [pd.to_datetime(date) for date in ["2009-04-01", "2020-01-01"]]
post_period = [pd.to_datetime(date) for date in ["2020-02-01", "2021-12-01"]]
ci = CausalImpact(pw_final['percentual'], pre_period, post_period, model_args={'fit_method': 'hmc'}, prior_level_sd=None)

# Result
print(ci.summary(output='report'))
#ci.plot(['original'])

# Export result CausalImpact analysis
dados = ci.inferences[['preds','preds_lower','preds_upper']]
dados['y'] = pw_final['percentual']
dados.to_excel('causal_impact_result.xlsx')